# Body Fat: post-processing, tables and figures

Run after the exact-enumeration notebook and the five training notebooks. Selects one fit per method and optimisation seed, applies the three importance-sampling corrections with $K=5000$ and scores them against the exact posterior. Results (including ESS) go to `bodyfat_POSTPROCESSING_tau2_1_final_S1_V20/<run id>`.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Mapping, Optional, Tuple
import hashlib
import json
import math
import os
import platform
import time
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

import matplotlib
matplotlib.use('Agg')
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator, FormatStrFormatter
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.special import betaln, expit, logsumexp
from scipy.stats import t as student_t
try:
    import arviz as az
except ImportError:
    az = None
try:
    from arviz_stats.base.array import array_stats as _arviz_array_stats
except ImportError:
    _arviz_array_stats = None

ROOT = Path.cwd().resolve()
BODYFAT_M0 = float(os.environ.get('BODYFAT_M0', '3'))   # 3 = main analysis; 1.5 or 6 = Appendix F
M0_TAG = f'{BODYFAT_M0:g}'.replace('.', 'p')
_SUFFIX = '' if BODYFAT_M0 == 3 else f'_m0_{M0_TAG}'
TRAINING_ROOT = ROOT / f'output_bodyfat_m0_{M0_TAG}_methodseeds_50_no_truth' / f'm0_{M0_TAG}'
OUTPUT_DIR = ROOT / f'bodyfat_POSTPROCESSING_tau2_1_final_S1_V20{_SUFFIX}'
EXACT_OUTPUT_DIR = ROOT / f'bodyfat_exact_primal_by_expected_size{_SUFFIX}'
METHOD_SEEDS = None  # Auto-detect shared seeds; optionally set (12345, 12346).
P, M0, TAU2, B_PI = 13, BODYFAT_M0, 1.0, 1.0
A_PI = B_PI * M0 / (P - M0)
WORKERS = 1
LR_GRID = (0.01, 0.05, 0.1, 0.2, 0.3)
ALPHA_GRID = (0.2, 0.5, 0.7, 0.9)
K_FIXED = 5000
PARETO_K_STOP = 0.7
IWMM_MAX_ITERS = 20
DAMPED_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)
DAMPED_MIN_K_IMPROVEMENT = 1e-12
PSIS_BASE_SEED = 1217
DTYPE, DEVICE = torch.float64, torch.device('cpu')
ADAPTED_PROB_EPS = 1e-12
PROBABILITY_METRIC_TOL = 1e-10
BATCH_SIZE = 250
EVAL_ELBO_SAMPLES = 2000
EVAL_ELBO_SEED = 2004
EVAL_BATCH_SIZE = 500
EVAL_TARGET_VERSION = 'bodyfat_tau2_1_full_constants_final_v1'
N_XSTAR = 100
N_MIXTURE_DRAWS = 20000
N_EXACT_REFERENCE_DRAWS = 50000
AR1_RHO = 0.8
TORCH_THREADS = 2
ANALYSIS_CODE_SHA256 = '8ff3605ee9363f870a3419f694228d5eacb6dc9eadb4b8bd4fc40e7c737a4c37'
METHODS = ['ELBO', 'VIMCO*', 'VIMCO* annealed', 'VIMCO-GM', 'VIMCO-GM annealed']
STAGES = ['Original VI', 'Original PSIS', 'Damped PSIS', 'IWMM PSIS']
METRICS = ['PIP RMSE vs exact', 'Mean excess marginal FKL vs exact', 'Pareto k-hat',
           'Exact posterior mass in 95% latent interval',
           'Exact-posterior expected 95% interval score']



## Collapsed target and PSIS


In [ ]:
def stable_seed(seed: int, method: str, setting_label: str, tag: str='') -> int:
    base = PSIS_BASE_SEED + int(seed)
    if tag in ('', '|ORDINARY_SHARED'):
        return base
    if tag.startswith('|DAMPED_PILOT'):
        stage = 1
    elif tag.startswith('|DAMPED_VALIDATE'):
        stage = 2
    elif tag.startswith('|IWMM_CANDIDATE'):
        stage = 3
    elif tag.startswith('|DAMPED_FINAL') or tag == '|IWMM_FINAL_EVAL':
        stage = 4
    else:
        raise ValueError(f'Unknown PSIS stage tag: {tag}')
    iteration = int(tag.split('|iter=')[-1].split('|')[0]) if '|iter=' in tag and stage <= 3 else 1
    if not 1 <= iteration <= IWMM_MAX_ITERS:
        raise ValueError(f'Invalid PSIS iteration: {iteration}')
    return base + 100000 * stage + 1000 * (iteration - 1)

def _normalise_logweights(logw: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    x = np.asarray(logw, dtype=np.float64).reshape(-1)
    if x.size == 0 or not np.all(np.isfinite(x)):
        raise FloatingPointError('Importance ratios must be finite and non-empty.')
    m = float(np.max(x))
    log_norm = m + math.log(float(np.sum(np.exp(x - m))))
    lw = x - log_norm
    w = np.exp(lw)
    w /= w.sum()
    return (lw, w)

def psis_smooth_logweights(raw_log_p_over_q: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
    x = np.asarray(raw_log_p_over_q, dtype=np.float64).reshape(-1)
    if x.size < 10 or not np.all(np.isfinite(x)):
        raise ValueError('PSIS requires at least 10 finite raw log importance ratios.')
    try:
        if az is not None and hasattr(az, 'psislw'):
            lw, khat = az.psislw(x, reff=1.0)
        elif _arviz_array_stats is not None:
            lw, khat = _arviz_array_stats.psislw(-x, r_eff=1.0, axis=-1)
        else:
            raise ImportError('A maintained ArviZ PSIS implementation is required. Install/update arviz.')
    except ValueError as exc:
        if 'tail values are the same' not in str(exc).lower():
            raise
        lw, w = _normalise_logweights(x)
        return (lw, w, 0.0)
    lw = np.asarray(lw, dtype=np.float64).reshape(-1)
    khat_arr = np.asarray(khat, dtype=np.float64).reshape(-1)
    if lw.shape != x.shape or khat_arr.size != 1:
        raise RuntimeError('Unexpected PSIS output shape.')
    _, w = _normalise_logweights(lw)
    return (lw, w, float(khat_arr[0]))

def validate_psis_backend_sign() -> None:
    x = np.linspace(-4.0, 4.0, 401, dtype=np.float64)
    _, w, _ = psis_smooth_logweights(x)
    if not np.isclose(w.sum(), 1.0, atol=1e-12):
        raise RuntimeError('PSIS backend sign self-test failed: weights do not sum to one.')
    if not float(w[-40:].sum()) > float(w[:40].sum()):
        raise RuntimeError('PSIS backend sign self-test failed: ratio ordering reversed.')

def sigmoid_eta_exact(eta: np.ndarray, label: str='eta') -> np.ndarray:
    eta = np.asarray(eta, dtype=np.float64).reshape(-1)
    if eta.size == 0 or not np.all(np.isfinite(eta)):
        raise ValueError(f'{label}: logits must be finite and non-empty.')
    rho = torch.sigmoid(torch.from_numpy(eta)).numpy()
    if np.any(rho <= 0.0) or np.any(rho >= 1.0):
        raise FloatingPointError(f'{label}: saved logits imply numerically degenerate probabilities. The original proposal is not clipped in the audited comparison.')
    return rho.astype(np.float64, copy=False)

def adapted_prob_to_eta(rho: np.ndarray, eps: float, label: str='adapted proposal') -> Tuple[np.ndarray, np.ndarray]:
    rho = np.asarray(rho, dtype=np.float64).reshape(-1)
    if rho.size == 0 or not np.all(np.isfinite(rho)):
        raise ValueError(f'{label}: probabilities must be finite and non-empty.')
    if np.any(rho < -1e-12) or np.any(rho > 1.0 + 1e-12):
        raise ValueError(f'{label}: probabilities outside [0,1].')
    eps = float(eps)
    if not 0.0 < eps < 0.5:
        raise ValueError('Adaptation clipping epsilon must lie in (0,0.5).')
    clipped = np.clip(rho, eps, 1.0 - eps)
    eta = np.log(clipped) - np.log1p(-clipped)
    if not np.all(np.isfinite(eta)):
        raise FloatingPointError(f'{label}: non-finite adapted logits.')
    return (clipped, eta)

@dataclass(frozen=True)
class TargetSpec:
    tau2: float
    a_pi: float
    b_pi: float = 1.0
    jitter: float = 1e-08
    primal_active_max: int = 80
    primal_batch_size: int = 64
    dual_batch_size: int = 8

class CollapsedLinearTarget:

    def __init__(self, X: np.ndarray, y: np.ndarray, spec: TargetSpec):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).reshape(-1)
        if X.ndim != 2 or y.ndim != 1 or X.shape[0] != len(y):
            raise ValueError('X must be n x p and y must be length n.')
        if not np.all(np.isfinite(X)) or not np.all(np.isfinite(y)):
            raise ValueError('X and y must be finite.')
        if X.shape[0] < 3 or X.shape[1] < 1:
            raise ValueError('Need n>=3 and p>=1.')
        if not (spec.tau2 > 0 and spec.a_pi > 0 and (spec.b_pi > 0) and (spec.jitter > 0)):
            raise ValueError('Target hyperparameters must be positive.')
        self.X = np.ascontiguousarray(X)
        self.y = np.ascontiguousarray(y)
        self.spec = spec
        self.n, self.p = X.shape
        self.nu = float(self.n - 1)
        self.prior_prec = 1.0 / float(spec.tau2)
        self.log_prior_prec = math.log(self.prior_prec)
        self.log_n = math.log(float(self.n))
        self.marginal_constant = float(math.lgamma(0.5 * self.nu) - 0.5 * self.nu * math.log(2.0 * math.pi))
        X_full = np.column_stack([np.ones(self.n, dtype=np.float64), self.X])
        self.XtX = torch.from_numpy(X_full.T @ X_full).to(dtype=DTYPE, device=DEVICE)
        self.Xty = torch.from_numpy(X_full.T @ self.y).to(dtype=DTYPE, device=DEVICE)
        self.yty = torch.tensor(float(self.y @ self.y), dtype=DTYPE, device=DEVICE)
        Xc = self.X - self.X.mean(axis=0, keepdims=True)
        yc = self.y - self.y.mean()
        self.Xc_T = torch.from_numpy(np.ascontiguousarray(Xc)).to(dtype=DTYPE, device=DEVICE).t().contiguous()
        self.yc = torch.from_numpy(np.ascontiguousarray(yc)).to(dtype=DTYPE, device=DEVICE)
        self.ycyc = torch.tensor(float(yc @ yc), dtype=DTYPE, device=DEVICE)
        self.I_n = torch.eye(self.n, dtype=DTYPE, device=DEVICE)
        k = np.arange(self.p + 1, dtype=np.float64)
        log_prior = betaln(k + spec.a_pi, self.p - k + spec.b_pi) - betaln(spec.a_pi, spec.b_pi)
        self.log_prior_by_size = torch.from_numpy(log_prior).to(dtype=DTYPE, device=DEVICE)
        self._regularisation: Dict[int, torch.Tensor] = {}

    def _reg_matrix(self, q: int) -> torch.Tensor:
        q = int(q)
        mat = self._regularisation.get(q)
        if mat is None:
            diag = torch.full((q,), self.prior_prec, dtype=DTYPE, device=DEVICE)
            diag[0] = 0.0
            mat = torch.diag(diag) + self.spec.jitter * torch.eye(q, dtype=DTYPE, device=DEVICE)
            self._regularisation[q] = mat
        return mat

    def _log_prior(self, G: torch.Tensor) -> torch.Tensor:
        k = G.sum(dim=1).to(torch.long)
        return self.log_prior_by_size[k]

    def _log_marginal_primal(self, G: torch.Tensor) -> torch.Tensor:
        B = int(G.shape[0])
        out = torch.empty(B, dtype=DTYPE, device=DEVICE)
        active_counts = G.sum(dim=1).to(torch.long)
        for count_t in torch.unique(active_counts):
            k = int(count_t.item())
            rows_all = torch.nonzero(active_counts == k, as_tuple=False).flatten()
            q = k + 1
            for start in range(0, int(rows_all.numel()), int(self.spec.primal_batch_size)):
                rows = rows_all[start:start + int(self.spec.primal_batch_size)]
                bsz = int(rows.numel())
                if k == 0:
                    idx = torch.zeros((bsz, 1), dtype=torch.long, device=DEVICE)
                else:
                    group = G.index_select(0, rows)
                    nz = torch.nonzero(group > 0.5, as_tuple=False)
                    slope_idx = nz[:, 1].reshape(bsz, k).to(torch.long) + 1
                    intercept = torch.zeros((bsz, 1), dtype=torch.long, device=DEVICE)
                    idx = torch.cat([intercept, slope_idx], dim=1)
                XtX_b = self.XtX[idx[:, :, None], idx[:, None, :]]
                Xty_b = self.Xty[idx]
                M = XtX_b + self._reg_matrix(q).unsqueeze(0)
                L = torch.linalg.cholesky(M)
                solved = torch.cholesky_solve(Xty_b[..., None], L).squeeze(-1)
                residual = self.yty - torch.sum(Xty_b * solved, dim=1)
                residual = torch.clamp(residual, min=self.spec.jitter)
                logdet_M = 2.0 * torch.sum(torch.log(torch.diagonal(L, dim1=-2, dim2=-1)), dim=1)
                out[rows] = self.marginal_constant + 0.5 * k * self.log_prior_prec - 0.5 * logdet_M - 0.5 * self.nu * torch.log(residual / 2.0)
        return out

    def _log_marginal_dual(self, G: torch.Tensor) -> torch.Tensor:
        B = int(G.shape[0])
        out = torch.empty(B, dtype=DTYPE, device=DEVICE)
        active_counts = G.sum(dim=1).to(torch.long)
        for count_t in torch.unique(active_counts):
            k = int(count_t.item())
            rows = torch.nonzero(active_counts == k, as_tuple=False).flatten()
            gsz = int(rows.numel())
            if k == 0:
                residual = torch.clamp(self.ycyc, min=self.spec.jitter)
                out[rows] = self.marginal_constant - 0.5 * self.log_n - 0.5 * self.nu * torch.log(residual / 2.0)
                continue
            group = G.index_select(0, rows)
            nz = torch.nonzero(group > 0.5, as_tuple=False)
            idx = nz[:, 1].reshape(gsz, k).to(torch.long)
            for start in range(0, gsz, int(self.spec.dual_batch_size)):
                current_rows = rows[start:start + int(self.spec.dual_batch_size)]
                current_idx = idx[start:start + int(self.spec.dual_batch_size)]
                bsz = int(current_idx.shape[0])
                XA = self.Xc_T[current_idx].transpose(1, 2).contiguous()
                Kmat = self.I_n.unsqueeze(0) + float(self.spec.tau2) * torch.bmm(XA, XA.transpose(1, 2))
                Kmat = Kmat + self.spec.jitter * self.I_n.unsqueeze(0)
                L = torch.linalg.cholesky(Kmat)
                yb = self.yc.expand(bsz, -1).unsqueeze(2)
                solved = torch.cholesky_solve(yb, L).squeeze(2)
                residual = torch.sum(self.yc.unsqueeze(0) * solved, dim=1)
                residual = torch.clamp(residual, min=self.spec.jitter)
                logdet_K = 2.0 * torch.sum(torch.log(torch.diagonal(L, dim1=-2, dim2=-1)), dim=1)
                out[current_rows] = self.marginal_constant - 0.5 * self.log_n - 0.5 * logdet_K - 0.5 * self.nu * torch.log(residual / 2.0)
        return out

    def log_joint(self, G: torch.Tensor) -> torch.Tensor:
        if G.ndim != 2 or int(G.shape[1]) != self.p:
            raise ValueError(f'G must have shape (B,{self.p}).')
        G = G.to(dtype=DTYPE, device=DEVICE)
        active = G.sum(dim=1)
        primal_mask = active <= int(self.spec.primal_active_max)
        out = torch.empty(int(G.shape[0]), dtype=DTYPE, device=DEVICE)
        if bool(primal_mask.any().item()):
            idx = torch.nonzero(primal_mask, as_tuple=False).flatten()
            out[idx] = self._log_marginal_primal(G.index_select(0, idx))
        if bool((~primal_mask).any().item()):
            idx = torch.nonzero(~primal_mask, as_tuple=False).flatten()
            out[idx] = self._log_marginal_dual(G.index_select(0, idx))
        return out + self._log_prior(G)

    def audit_primal_dual(self, G: np.ndarray, atol: float=1e-05) -> float:
        Gt = torch.as_tensor(np.asarray(G, dtype=np.float64), dtype=DTYPE, device=DEVICE)
        with torch.no_grad():
            a = self._log_marginal_primal(Gt)
            b = self._log_marginal_dual(Gt)
        d = (a - b).cpu().numpy()
        gap = float(np.max(np.abs(d))) if len(d) else 0.0
        if gap > float(atol):
            raise RuntimeError(f'Primal/dual collapsed-target audit failed: max absolute gap={gap:.3e}.')
        return gap

@dataclass
class DrawResult:
    eta: np.ndarray
    rho: np.ndarray
    G: np.ndarray
    raw_logw: np.ndarray
    raw_weights: np.ndarray
    psis_weights: np.ndarray
    pip: np.ndarray
    khat: float
    raw_ess: float
    psis_ess: float
    seed: int
    K: int

    def diagnostics(self) -> Dict[str, Any]:
        return {'K': int(self.K), 'seed': int(self.seed), 'Pareto_k': float(self.khat), 'raw_ESS': float(self.raw_ess), 'PSIS_ESS': float(self.psis_ess), 'PSIS_ESS_fraction': float(self.psis_ess / self.K), 'max_PSIS_weight': float(np.max(self.psis_weights)), 'mean_sampled_model_size': float(np.mean(self.G.sum(axis=1)))}

def log_q_eta(G: torch.Tensor, eta_t: torch.Tensor) -> torch.Tensor:
    return (G * eta_t[None, :] - F.softplus(eta_t)[None, :]).sum(dim=1)

def draw_and_score(target: CollapsedLinearTarget, eta: np.ndarray, draw_seed: int, K: int=K_FIXED, batch_size: int=1000, route_audit_draws: int=20) -> DrawResult:
    if int(K) != K_FIXED:
        raise ValueError(f'This comparison fixes K={K_FIXED}; received K={K}.')
    eta = np.asarray(eta, dtype=np.float64).reshape(-1)
    rho = sigmoid_eta_exact(eta, label='proposal eta')
    if len(eta) != target.p:
        raise ValueError(f'Proposal length {len(eta)} does not match p={target.p}.')
    eta_t = torch.from_numpy(eta).to(dtype=DTYPE, device=DEVICE)
    rho_t = torch.from_numpy(rho).to(dtype=DTYPE, device=DEVICE)
    torch.manual_seed(int(draw_seed))
    G_chunks = []
    lw_chunks = []
    remaining = int(K)
    with torch.no_grad():
        while remaining > 0:
            b = min(int(batch_size), remaining)
            G = torch.bernoulli(rho_t.expand(b, -1))
            logp = target.log_joint(G)
            logq = log_q_eta(G, eta_t)
            lw = logp - logq
            if not bool(torch.isfinite(lw).all().item()):
                raise FloatingPointError('Non-finite raw log importance ratios.')
            G_chunks.append(G.to(dtype=torch.uint8).cpu().numpy())
            lw_chunks.append(lw.cpu().numpy().astype(np.float64))
            remaining -= b
    G_np = np.concatenate(G_chunks, axis=0)
    raw_logw = np.concatenate(lw_chunks, axis=0)
    _, w_raw = _normalise_logweights(raw_logw)
    _, w_psis, khat = psis_smooth_logweights(raw_logw)
    pip = np.asarray(w_psis @ G_np, dtype=np.float64)
    raw_ess = float(1.0 / np.sum(w_raw * w_raw))
    psis_ess = float(1.0 / np.sum(w_psis * w_psis))
    n_audit = min(int(route_audit_draws), len(G_np))
    if n_audit >= 2:
        target.audit_primal_dual(G_np[:n_audit])
    return DrawResult(eta=eta.copy(), rho=rho.copy(), G=G_np, raw_logw=raw_logw, raw_weights=w_raw, psis_weights=w_psis, pip=np.clip(pip, 0.0, 1.0), khat=float(khat), raw_ess=raw_ess, psis_ess=psis_ess, seed=int(draw_seed), K=int(K))

def _choose_damped_candidate(candidate: Mapping[float, DrawResult]) -> Tuple[float, str]:
    base = candidate[0.0]
    finite_nonzero = [lam for lam in DAMPED_GRID if lam != 0.0 and np.isfinite(candidate[lam].khat)]
    if np.isfinite(base.khat):
        if not finite_nonzero:
            return (0.0, 'no_finite_nonzero_candidate_k')
        best = min(finite_nonzero, key=lambda lam: (candidate[lam].khat, -candidate[lam].psis_ess, lam))
        if candidate[best].khat <= base.khat - DAMPED_MIN_K_IMPROVEMENT:
            return (float(best), 'validation_k_improved_over_omega0')
        return (0.0, 'no_material_validation_k_improvement')
    if finite_nonzero:
        best = min(finite_nonzero, key=lambda lam: (candidate[lam].khat, -candidate[lam].psis_ess, lam))
        return (float(best), 'baseline_k_nonfinite__finite_damped_candidate')
    return (0.0, 'no_finite_validation_k__keep_original_q')

def run_damped(target: CollapsedLinearTarget, eta_vi: np.ndarray, method: str, run_seed: int, setting_label: str, adapted_prob_eps: float, batch_size: int, ordinary_override: Optional[DrawResult]=None) -> Dict[str, Any]:
    eta_vi = np.asarray(eta_vi, dtype=np.float64).reshape(-1)
    rho_vi = sigmoid_eta_exact(eta_vi, 'original q_VI eta')
    if ordinary_override is None:
        ordinary_seed = stable_seed(run_seed, method, setting_label, '|ORDINARY_SHARED')
        ordinary = draw_and_score(target, eta_vi, ordinary_seed, batch_size=batch_size)
    else:
        ordinary = ordinary_override
        if ordinary.K != K_FIXED:
            raise RuntimeError('ordinary_override must use fixed K=5000.')
        if not np.array_equal(np.asarray(ordinary.eta), eta_vi):
            raise RuntimeError('ordinary_override proposal does not match eta_vi.')
    current_eta = eta_vi.copy()
    current_rho = rho_vi.copy()
    current_k = float(ordinary.khat)
    triggered = bool(not np.isfinite(current_k) or current_k > PARETO_K_STOP)
    success = bool(np.isfinite(current_k) and current_k <= PARETO_K_STOP)
    stop_reason = 'ordinary_k_at_or_below_0.7' if success else 'adaptation_not_started'
    history = []
    accepted_iterations = 0
    accepted_omegas = []
    first_pilot = None
    first_rho_hat = np.asarray(ordinary.pip, dtype=np.float64)
    max_iters = int(IWMM_MAX_ITERS)
    if triggered:
        for it in range(1, max_iters + 1):
            if np.isfinite(current_k) and current_k <= PARETO_K_STOP:
                success = True
                stop_reason = 'Pareto_k_at_or_below_0.7'
                break
            if it == 1:
                pilot_tag = '|DAMPED_PILOT'
                validation_tag = '|DAMPED_VALIDATE'
            else:
                pilot_tag = f'|DAMPED_PILOT|iter={it}'
                validation_tag = f'|DAMPED_VALIDATE|iter={it}'
            pilot_seed = stable_seed(run_seed, method, setting_label, pilot_tag)
            pilot = draw_and_score(target, current_eta, pilot_seed, batch_size=batch_size)
            rho_hat = np.asarray(pilot.pip, dtype=np.float64)
            if it == 1:
                first_pilot = pilot
                first_rho_hat = rho_hat.copy()
            validation_seed = stable_seed(run_seed, method, setting_label, validation_tag)
            validation: Dict[float, DrawResult] = {}
            candidate_eta: Dict[float, np.ndarray] = {}
            candidate_rho: Dict[float, np.ndarray] = {}
            for omega in DAMPED_GRID:
                omega = float(omega)
                raw_rho = (1.0 - omega) * current_rho + omega * rho_hat
                if omega == 0.0:
                    rho = current_rho.copy()
                    eta = current_eta.copy()
                else:
                    rho, eta = adapted_prob_to_eta(raw_rho, adapted_prob_eps, f'damped iteration={it}, omega={omega:g}')
                candidate_rho[omega] = rho
                candidate_eta[omega] = eta
                validation[omega] = draw_and_score(target, eta, validation_seed, batch_size=batch_size)
            selected_omega, reason = _choose_damped_candidate(validation)
            baseline_k = float(validation[0.0].khat)
            for omega in DAMPED_GRID:
                omega = float(omega)
                d = validation[omega].diagnostics()
                d.update({'iteration': int(it), 'omega': omega, 'selected': bool(np.isclose(omega, selected_omega) and selected_omega != 0.0), 'baseline_k': baseline_k, 'selection_reason': reason, 'pilot_seed': int(pilot.seed), 'validation_seed': int(validation[omega].seed), 'mean_abs_probability_change': float(np.mean(np.abs(candidate_rho[omega] - current_rho)))})
                history.append(d)
            if selected_omega == 0.0:
                current_k = baseline_k
                stop_reason = reason
                success = bool(np.isfinite(current_k) and current_k <= PARETO_K_STOP)
                break
            current_rho = candidate_rho[selected_omega].copy()
            current_eta = candidate_eta[selected_omega].copy()
            current_k = float(validation[selected_omega].khat)
            accepted_iterations += 1
            accepted_omegas.append(float(selected_omega))
            if np.isfinite(current_k) and current_k <= PARETO_K_STOP:
                success = True
                stop_reason = 'Pareto_k_at_or_below_0.7'
                break
        else:
            stop_reason = 'maximum_iterations_reached'
            success = bool(np.isfinite(current_k) and current_k <= PARETO_K_STOP)
    last_omega = float(accepted_omegas[-1]) if accepted_omegas else 0.0
    if accepted_iterations <= 1:
        final_tag = f'|DAMPED_FINAL|omega={last_omega:.6g}'
    else:
        omega_path = ','.join((f'{x:.6g}' for x in accepted_omegas))
        final_tag = f'|DAMPED_FINAL_ITERATIVE|iters={accepted_iterations}|omegas={omega_path}'
    final_seed = stable_seed(run_seed, method, setting_label, final_tag)
    final = draw_and_score(target, current_eta, final_seed, batch_size=batch_size)
    return {'method': method, 'run_seed': int(run_seed), 'setting_label': str(setting_label), 'rho_vi': rho_vi, 'eta_vi': eta_vi, 'ordinary': ordinary, 'triggered': triggered, 'pilot': first_pilot, 'pilot_weighted_pip': first_rho_hat, 'selected_omega': last_omega, 'selection_reason': stop_reason, 'selected_rho': current_rho, 'selected_eta': current_eta, 'final': final, 'history': history, 'accepted_iterations': int(accepted_iterations), 'accepted_omegas': tuple(accepted_omegas), 'adaptation_success': bool(success), 'stop_reason': str(stop_reason), 'selection_khat': float(current_k), 'final_seed_tag': str(final_tag)}

def run_iwmm(target: CollapsedLinearTarget, eta_vi: np.ndarray, method: str, run_seed: int, setting_label: str, adapted_prob_eps: float, batch_size: int, ordinary_override: Optional[DrawResult]=None) -> Dict[str, Any]:
    eta_vi = np.asarray(eta_vi, dtype=np.float64).reshape(-1)
    rho_vi = sigmoid_eta_exact(eta_vi, 'original q_VI eta')
    if ordinary_override is None:
        ordinary_seed = stable_seed(run_seed, method, setting_label, '|ORDINARY_SHARED')
        ordinary = draw_and_score(target, eta_vi, ordinary_seed, batch_size=batch_size)
    else:
        ordinary = ordinary_override
        if ordinary.K != K_FIXED:
            raise RuntimeError('ordinary_override must use fixed K=5000.')
        if not np.array_equal(np.asarray(ordinary.eta), eta_vi):
            raise RuntimeError('ordinary_override proposal does not match eta_vi.')
    current_eta = eta_vi.copy()
    current_rho = rho_vi.copy()
    current_eval = ordinary
    current_k = float(ordinary.khat)
    triggered = bool(not np.isfinite(current_k) or current_k > PARETO_K_STOP)
    success = bool(np.isfinite(current_k) and current_k <= PARETO_K_STOP)
    stop_reason = 'ordinary_k_at_or_below_0.7' if success else 'adaptation_not_started'
    history = []
    accepted = 0
    if triggered:
        for it in range(1, IWMM_MAX_ITERS + 1):
            if np.isfinite(current_k) and current_k <= PARETO_K_STOP:
                success = True
                stop_reason = 'Pareto_k_at_or_below_0.7'
                break
            proposed_rho_raw = np.asarray(current_eval.pip, dtype=np.float64)
            proposed_rho, proposed_eta = adapted_prob_to_eta(proposed_rho_raw, adapted_prob_eps, f'IWMM iteration {it}')
            candidate_seed = stable_seed(run_seed, method, setting_label, f'|IWMM_CANDIDATE|iter={it}')
            candidate = draw_and_score(target, proposed_eta, candidate_seed, batch_size=batch_size)
            improves = bool(np.isfinite(candidate.khat) and (not np.isfinite(current_k) or candidate.khat < current_k - 1e-12))
            history.append({'iteration': int(it), 'current_k_before': float(current_k), 'candidate_k': float(candidate.khat), 'candidate_PSIS_ESS': float(candidate.psis_ess), 'accepted': improves, 'candidate_seed': int(candidate.seed), 'mean_abs_probability_change': float(np.mean(np.abs(proposed_rho - current_rho)))})
            if not improves:
                stop_reason = 'candidate_did_not_reduce_Pareto_k'
                break
            current_eta = proposed_eta
            current_rho = proposed_rho
            current_eval = candidate
            current_k = float(candidate.khat)
            accepted += 1
            if np.isfinite(current_k) and current_k <= PARETO_K_STOP:
                success = True
                stop_reason = 'Pareto_k_at_or_below_0.7'
                break
        else:
            stop_reason = 'maximum_iterations_reached'
            success = bool(np.isfinite(current_k) and current_k <= PARETO_K_STOP)
    final_seed = stable_seed(run_seed, method, setting_label, '|IWMM_FINAL_EVAL')
    final = draw_and_score(target, current_eta, final_seed, batch_size=batch_size)
    return {'method': method, 'run_seed': int(run_seed), 'setting_label': str(setting_label), 'rho_vi': rho_vi, 'eta_vi': eta_vi, 'ordinary': ordinary, 'triggered': triggered, 'accepted_iterations': int(accepted), 'adaptation_success': success, 'stop_reason': stop_reason, 'selected_rho': current_rho, 'selected_eta': current_eta, 'selected_adaptation_k': float(current_k), 'final': final, 'history': history}



## Evaluation and reporting helpers


In [ ]:
def _eval_elbo_2000(target, eta: np.ndarray, batch_size: int):
    eta = np.asarray(eta, dtype=np.float64).reshape(-1)
    rho = sigmoid_eta_exact(eta, label='evaluated-ELBO eta')
    eta_t = torch.from_numpy(eta).to(dtype=DTYPE, device=DEVICE)
    rho_t = torch.from_numpy(rho).to(dtype=DTYPE, device=DEVICE)
    torch.manual_seed(int(EVAL_ELBO_SEED))
    vals = []
    active_sum = 0.0
    audit_G = None
    remaining = EVAL_ELBO_SAMPLES
    with torch.no_grad():
        while remaining > 0:
            b = min(int(batch_size), remaining)
            G = torch.bernoulli(rho_t.expand(b, -1))
            if audit_G is None:
                audit_G = G[:min(2, b)].detach().cpu().numpy()
            v = target.log_joint(G) - log_q_eta(G, eta_t)
            if not bool(torch.isfinite(v).all().item()):
                raise FloatingPointError('Non-finite evaluated-ELBO draw.')
            vals.append(v.cpu().numpy().astype(np.float64))
            active_sum += float(G.sum().item())
            remaining -= b
    x = np.concatenate(vals)
    if len(x) != int(EVAL_ELBO_SAMPLES):
        raise RuntimeError('Evaluated-ELBO sample count mismatch.')
    if audit_G is not None and len(audit_G) >= 1:
        target.audit_primal_dual(audit_G, atol=2e-05)
    sd = float(np.std(x, ddof=1))
    return {'evaluated_elbo_2000': float(np.mean(x)), 'evaluated_elbo_2000 MCSE': float(sd / math.sqrt(len(x))), 'evaluated_elbo_2000 sd': sd, 'evaluated_elbo_2000 mean sampled active size': float(active_sum / len(x)), 'evaluated_elbo_2000 target version': EVAL_TARGET_VERSION}

def _select_best_mcse(group):
    g = group.copy()
    g['evaluated_elbo_2000'] = _num(g['evaluated_elbo_2000'])
    g['evaluated_elbo_2000 MCSE'] = _num(g['evaluated_elbo_2000 MCSE'])
    g = g.dropna(subset=['evaluated_elbo_2000', 'evaluated_elbo_2000 MCSE']).sort_values('evaluated_elbo_2000', ascending=False, kind='mergesort').reset_index(drop=True)
    if g.empty:
        raise RuntimeError('No valid evaluated ELBO values in a method/seed group.')
    top = g.iloc[0]
    Lmax = float(top['evaluated_elbo_2000'])
    mmax = float(top['evaluated_elbo_2000 MCSE'])
    gaps = Lmax - _num(g['evaluated_elbo_2000']).to_numpy(float)
    lim = 2.0 * np.sqrt(mmax * mmax + _num(g['evaluated_elbo_2000 MCSE']).to_numpy(float) ** 2)
    tied = g.loc[gaps <= lim].copy()
    tied['_m'] = _num(tied['evaluated_elbo_2000 MCSE'])
    tied['_L'] = _num(tied['evaluated_elbo_2000'])
    tied = tied.sort_values(['_m', '_L'], ascending=[True, False], kind='mergesort')
    out = tied.iloc[0].drop(labels=['_m', '_L']).copy()
    out['raw_highest_elbo'] = Lmax
    out['number_of_mcse_tied_settings'] = int(len(tied))
    out['selection_rule'] = 'all_within_2_combined_mcse_then_smallest_mcse'
    return out

def _posterior_components(G, X, y, Xstar, tau2=1.0, observation_noise=False):
    G = np.asarray(G, np.uint8)
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    Xstar = np.asarray(Xstar, float)
    n, p = X.shape
    nu = float(n - 1)
    xbar = X.mean(0)
    ybar = float(y.mean())
    Xc = X - xbar
    yc = y - ybar
    ycyc = float(yc @ yc)
    loc = np.empty((len(G), len(Xstar)), float)
    scale2 = np.empty_like(loc)
    packed = np.packbits(G, axis=1)
    groups = {}
    for i in range(len(G)):
        groups.setdefault(packed[i].tobytes(), []).append(i)
    for pos in groups.values():
        pos = np.asarray(pos, int)
        idx = np.flatnonzero(G[pos[0]])
        k = len(idx)
        if k == 0:
            m = np.empty(0)
            L = None
            rss = max(ycyc, 1e-08)
        else:
            XA = Xc[:, idx]
            M = XA.T @ XA + (1.0 / float(tau2) + 1e-08) * np.eye(k)
            rhs = XA.T @ yc
            L = np.linalg.cholesky(M)
            m = np.linalg.solve(L.T, np.linalg.solve(L, rhs))
            rss = max(float(ycyc - rhs @ m), 1e-08)
        if k == 0:
            mu = np.full(len(Xstar), ybar)
            lev = np.full(len(Xstar), 1.0 / n)
        else:
            D = Xstar[:, idx] - xbar[idx][None, :]
            mu = ybar + D @ m
            V = np.linalg.solve(L, D.T)
            lev = 1.0 / n + np.sum(V * V, axis=0)
        if observation_noise:
            lev = 1.0 + lev
        loc[pos] = mu[None, :]
        scale2[pos] = rss / nu * lev[None, :]
    return (loc, scale2, nu)

def _interval_metrics(loc, scale2, weights, df, seed, truth=None):
    w = np.asarray(weights, float)
    w = w / w.sum()
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(len(w), size=N_MIXTURE_DRAWS, replace=True, p=w)
    draws = loc[idx] + np.sqrt(scale2[idx]) * rng.standard_t(df=float(df), size=(N_MIXTURE_DRAWS, loc.shape[1]))
    lo, hi = np.quantile(draws, [0.025, 0.975], axis=0)
    ans = {'interval_width_95': float(np.mean(hi - lo))}
    if truth is not None:
        ans['coverage_95'] = float(np.mean((truth >= lo) & (truth <= hi)))
        ans['coverage_deviation_95'] = float(abs(ans['coverage_95'] - 0.95))
        ans['interval_score_95'] = _interval_score(lo, hi, truth, alpha=0.05)
    return (ans, lo, hi)

def clean_float(x):
    return str(float(x)).replace('.', 'p').replace('-', 'm')

def method_token(method):
    return method.replace('*', 'star').replace('-', '_').replace(' ', '_').lower()

def expected_grid(method):
    allocation = 'S_300_N_1' if method == 'ELBO' else 'S_1_N_300'
    if method == 'ELBO':
        return [(lr, 'ELBO', 300, 1, f'elbo_lr_{clean_float(lr)}') for lr in LR_GRID]
    if method in ('VIMCO*', 'VIMCO-GM'):
        prefix = 'star' if method == 'VIMCO*' else 'gm'
        return [(lr, alpha, 1, 300,
                 f'{prefix}_alpha_{clean_float(alpha)}_lr_{clean_float(lr)}_{allocation}')
                for alpha in ALPHA_GRID for lr in LR_GRID]
    prefix = 'star_annealed' if method == 'VIMCO* annealed' else 'gm_annealed'
    return [(lr, 'annealed', 1, 300, f'{prefix}_lr_{clean_float(lr)}_{allocation}')
            for lr in LR_GRID]

def _evaluation_fields(record):
    try:
        mean = float(record['evaluated_elbo_2000'])
        mcse = float(record['evaluated_elbo_2000 MCSE'])
        version = str(record['evaluated_elbo_2000 target version'])
    except (KeyError, TypeError, ValueError):
        return None
    if not np.isfinite(mean) or not np.isfinite(mcse) or mcse < 0 or version != EVAL_TARGET_VERSION:
        return None
    return {key: value for key, value in record.items()
            if key.startswith('evaluated_elbo_2000') or key.startswith('search_')}

def _sanitize_probability_vector(x: np.ndarray, label: str='probability vector', tol: float=PROBABILITY_METRIC_TOL) -> np.ndarray:
    a = np.asarray(x, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(a)):
        raise ValueError(f'{label} must be finite.')
    lo = float(np.min(a)) if a.size else 0.0
    hi = float(np.max(a)) if a.size else 1.0
    if lo < -tol or hi > 1.0 + tol:
        raise ValueError(f'{label} lies outside [0,1] beyond numerical tolerance {tol:g}: min={lo:.17g}, max={hi:.17g}.')
    return np.clip(a, 0.0, 1.0)

def forward_kl_product_bernoulli(reference_pip: np.ndarray, approx_pip: np.ndarray) -> float:
    pi_raw = np.asarray(reference_pip, dtype=np.float64).reshape(-1)
    rho_raw = np.asarray(approx_pip, dtype=np.float64).reshape(-1)
    if pi_raw.shape != rho_raw.shape:
        raise ValueError('FKL inputs must have the same shape.')
    pi = _sanitize_probability_vector(pi_raw, 'FKL reference PIPs')
    rho = _sanitize_probability_vector(rho_raw, 'FKL approximate PIPs')
    eps = 1e-12
    r = np.clip(rho, eps, 1.0 - eps)
    term1 = np.where(pi > 0.0, pi * (np.log(np.clip(pi, eps, 1.0)) - np.log(r)), 0.0)
    one_pi = 1.0 - pi
    term0 = np.where(one_pi > 0.0, one_pi * (np.log(np.clip(one_pi, eps, 1.0)) - np.log1p(-r)), 0.0)
    out = float(np.sum(term1 + term0))
    if not np.isfinite(out) or out < -1e-10:
        raise RuntimeError(f'Computed FKL is invalid: {out}.')
    return max(out, 0.0)

def _num(x):
    return pd.to_numeric(x, errors='coerce')

def _rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    return float(np.sqrt(np.mean((a - b) ** 2)))

def _ci95(values):
    x = _num(pd.Series(values)).dropna().to_numpy(float)
    n = len(x)
    if n == 0:
        return dict(n=0, mean=np.nan, sd=np.nan, se=np.nan, lo=np.nan, hi=np.nan)
    mean = float(np.mean(x))
    if n == 1:
        return dict(n=1, mean=mean, sd=np.nan, se=np.nan, lo=np.nan, hi=np.nan)
    sd = float(np.std(x, ddof=1))
    se = sd / math.sqrt(n)
    crit = float(student_t.ppf(0.975, n - 1)) if student_t is not None else 1.96
    return dict(n=n, mean=mean, sd=sd, se=se, lo=mean - crit * se, hi=mean + crit * se)

def _interval_score(lo, hi, y, alpha=0.05):
    lo = np.asarray(lo, float)
    hi = np.asarray(hi, float)
    y = np.asarray(y, float)
    score = hi - lo + 2.0 / alpha * (lo - y) * (y < lo) + 2.0 / alpha * (y - hi) * (y > hi)
    return float(np.mean(score))

def _reference_scatter_spec(profile_slug: str):
    if profile_slug == 'bodyfat':
        return {'ref_col': 'reference_pip', 'xlabel': 'Exact posterior inclusion probability', 'ylabel': 'Estimated posterior inclusion probability', 'title': 'Posterior inclusion probabilities relative to the exact reference', 'filename': 'figure_scatter_selected_pips_vs_exact_reference.png'}
    if profile_slug == 'bodyfat':
        return {'ref_col': 'reference_pip', 'xlabel': 'Exact posterior inclusion probability', 'ylabel': 'Estimated posterior inclusion probability', 'title': 'Posterior inclusion probabilities relative to exact enumeration', 'filename': 'figure_scatter_selected_pips_vs_exact_reference.png'}
    return None

def make_reference_scatter_grid(scatter_df: pd.DataFrame, out_dir: Path, profile_slug: str):
    spec = _reference_scatter_spec(profile_slug)
    if spec is None or scatter_df is None or scatter_df.empty:
        return None
    out_dir = Path(out_dir)
    plot_dir = out_dir / 'plots'
    plot_dir.mkdir(parents=True, exist_ok=True)
    df = scatter_df.copy()
    df = df.dropna(subset=['reference_pip', 'method_pip', 'method'])
    if df.empty:
        return None
    df['reference_pip'] = _num(df['reference_pip'])
    df['method_pip'] = _num(df['method_pip'])
    df = df[np.isfinite(df['reference_pip']) & np.isfinite(df['method_pip'])].copy()
    if df.empty:
        return None
    method_order = [m for m in METHODS if m in set(df['method'].astype(str))]
    if 'varbvs' in set(df['method'].astype(str)):
        method_order = method_order + ['varbvs']
    if not method_order:
        return None
    ncols = 3
    nrows = 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(15.5, 9.2), squeeze=False)
    axes_flat = axes.ravel()
    for ax, method in zip(axes_flat, method_order):
        d = df[df['method'].astype(str) == method].copy()
        if d.empty:
            ax.axis('off')
            continue
        ax.scatter(d['reference_pip'], d['method_pip'], s=14, alpha=0.38)
        ax.plot([0, 1], [0, 1], '--', linewidth=1.0)
        ax.set_xlim(-0.03, 1.03)
        ax.set_ylim(-0.03, 1.03)
        ax.set_xlabel(spec['xlabel'])
        ax.set_ylabel(spec['ylabel'])
        ax.set_title(method)
        ax.grid(alpha=0.22)
    for ax in axes_flat[len(method_order):]:
        ax.axis('off')
    fig.suptitle(spec['title'], fontsize=15, y=0.995)
    fig.tight_layout(rect=[0, 0.02, 1, 0.97])
    scatter_path = plot_dir / spec['filename']
    fig.savefig(scatter_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return scatter_path

def make_stage_scatter_2x2(scatter_stage_df: pd.DataFrame, out_dir: Path, profile_slug: str, focus_method: str='VIMCO-GM annealed'):
    spec = _reference_scatter_spec(profile_slug)
    if spec is None or scatter_stage_df is None or scatter_stage_df.empty:
        return None
    out_dir = Path(out_dir)
    plot_dir = out_dir / 'plots'
    plot_dir.mkdir(parents=True, exist_ok=True)
    df = scatter_stage_df.copy()
    req = ['reference_pip', 'method_pip', 'stage', 'method']
    missing = [c for c in req if c not in df.columns]
    if missing:
        return None
    df = df.dropna(subset=['reference_pip', 'method_pip', 'stage', 'method']).copy()
    if df.empty:
        return None
    df['reference_pip'] = _num(df['reference_pip'])
    df['method_pip'] = _num(df['method_pip'])
    df = df[np.isfinite(df['reference_pip']) & np.isfinite(df['method_pip'])].copy()
    if df.empty:
        return None
    methods_present = list(df['method'].astype(str).drop_duplicates())
    chosen_method = focus_method if focus_method in methods_present else methods_present[0] if methods_present else None
    if chosen_method is None:
        return None
    d0 = df[df['method'].astype(str) == chosen_method].copy()
    if d0.empty:
        return None
    stage_order = ['Original VI', 'Original PSIS', 'Damped PSIS', 'IWMM PSIS']
    fig, axes = plt.subplots(2, 2, figsize=(12.8, 10.0), squeeze=False)
    axes_flat = axes.ravel()
    for ax, stage in zip(axes_flat, stage_order):
        d = d0[d0['stage'].astype(str) == stage].copy()
        if d.empty:
            ax.axis('off')
            continue
        ax.scatter(d['reference_pip'], d['method_pip'], s=14, alpha=0.38)
        ax.plot([0, 1], [0, 1], '--', linewidth=1.0)
        ax.set_xlim(-0.03, 1.03)
        ax.set_ylim(-0.03, 1.03)
        ax.set_xlabel(spec['xlabel'])
        ax.set_ylabel(spec['ylabel'])
        ax.set_title(stage)
        ax.grid(alpha=0.22)
    fig.suptitle(f'{chosen_method}: posterior inclusion probabilities across post-processing stages', fontsize=15, y=0.995)
    fig.tight_layout(rect=[0, 0.02, 1, 0.97])
    scatter_path = plot_dir / 'figure_scatter_postprocessing_stages_vs_reference.png'
    fig.savefig(scatter_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return scatter_path

def _style_maps(stages):
    stage_offsets = {'Original VI': -0.27, 'Original PSIS': -0.09, 'IWMM PSIS': 0.09, 'Iterative Damped PSIS': 0.27}
    stage_markers = {'Original VI': 'o', 'Original PSIS': 's', 'Iterative Damped PSIS': 'D', 'IWMM PSIS': '^'}
    default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    all_stage_order = ['Original VI', 'Original PSIS', 'IWMM PSIS', 'Iterative Damped PSIS']
    stage_colors = {s: default_colors[i] for i, s in enumerate(all_stage_order)}
    return (stage_offsets, stage_markers, stage_colors)

def _draw_panel(ax, dd, metric, methods, stages, stage_offsets, stage_markers, stage_colors, title):
    method_index = {m: i for i, m in enumerate(methods)}
    base_y = np.arange(len(methods), dtype=float)
    for stage in stages:
        ds = dd[dd['stage'].astype(str) == stage].copy()
        if ds.empty:
            continue
        ds['_order'] = ds['method'].map(method_index)
        ds = ds.sort_values('_order').set_index('method').reindex(methods)
        x = pd.to_numeric(ds['oriented_mean_difference_left_is_better'], errors='coerce').to_numpy(float)
        lo = pd.to_numeric(ds['oriented_95% interval lower'], errors='coerce').to_numpy(float)
        hi = pd.to_numeric(ds['oriented_95% interval upper'], errors='coerce').to_numpy(float)
        y = base_y + stage_offsets[stage]
        ok = np.isfinite(x) & np.isfinite(lo) & np.isfinite(hi)
        if np.any(ok):
            ax.errorbar(x[ok], y[ok], xerr=[x[ok] - lo[ok], hi[ok] - x[ok]], fmt=stage_markers[stage], color=stage_colors[stage], capsize=3.2, elinewidth=1.15, linewidth=0, markersize=5.8, label=stage)
        points_only = np.isfinite(x) & ~ok
        if np.any(points_only):
            ax.plot(x[points_only], y[points_only], linestyle="none",
                marker=stage_markers[stage], color=stage_colors[stage], markersize=5.8,
                label=stage if not np.any(ok) else "_nolegend_")
    ax.axvline(0.0, linestyle='--', linewidth=1.05, color='0.25', zorder=0)
    ax.set_yticks(base_y)
    ax.set_yticklabels(methods)
    ax.invert_yaxis()
    ax.set_xlabel('Paired difference vs ELBO')
    ax.grid(axis='x', alpha=0.2)
    finite_bounds = []
    for col in ['oriented_95% interval lower', 'oriented_95% interval upper']:
        vals = pd.to_numeric(dd[col], errors='coerce').dropna()
        finite_bounds.extend(vals.tolist())
    if finite_bounds:
        bound = float(np.max(np.abs(np.asarray(finite_bounds, dtype=float))))
        bound = 1.12 * bound if bound > 0 else 1.0
        ax.set_xlim(-bound, bound)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))

def _panel_heading(ax, letter, title):
    title = str(title).replace('\n', ' ').strip()
    ax.set_title(f'({letter}) {title}', loc='left', pad=8, fontsize=10.0, fontweight='normal', wrap=False)

def _finish_legend(fig, axes, stages, y=0.02, ncol=None):
    handle_map = {}
    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()
        for h, lab in zip(handles, labels):
            if lab not in handle_map:
                handle_map[lab] = h
    legend_stages = [s for s in stages if s in handle_map]
    if ncol is None:
        ncol = len(legend_stages)
    fig.legend([handle_map[s] for s in legend_stages], legend_stages, loc='lower center', ncol=ncol, frameon=False, bbox_to_anchor=(0.5, y))

def _find_metric(metric_values, exact_candidates, required_tokens):
    for name in exact_candidates:
        if name in metric_values:
            return name
    for name in metric_values:
        low = name.lower()
        if all((tok.lower() in low for tok in required_tokens)):
            return name
    raise RuntimeError('Could not find required metric.\nRequired tokens: {}\nAvailable metrics:\n  - {}'.format(required_tokens, '\n  - '.join(metric_values)))

def plot_paired_v20(paired, out):
    paired = paired.copy()
    paired['stage'] = paired['stage'].replace({'Damped PSIS': 'Iterative Damped PSIS'})
    METHODS = ['VIMCO*', 'VIMCO* annealed', 'VIMCO-GM', 'VIMCO-GM annealed']
    METHODS = [m for m in METHODS if m in set(paired['method'].astype(str))]
    STAGES = ['Original VI', 'Original PSIS', 'IWMM PSIS', 'Iterative Damped PSIS']
    STAGES = [s for s in STAGES if s in set(paired['stage'].astype(str))]
    metric_values = list(dict.fromkeys(paired['metric'].astype(str).tolist()))
    RMSE = _find_metric(metric_values, ['PIP RMSE vs exact'], ['rmse', 'exact'])
    FKL = _find_metric(metric_values, ['Mean excess marginal FKL vs exact', 'Excess FKL vs exact'], ['kl', 'exact'])
    INTERVAL = _find_metric(metric_values, ['Exact-posterior expected 95% interval score'], ['interval', 'score'])
    PLOT_METRICS = [RMSE, FKL, INTERVAL]
    PANEL_TITLES = {RMSE: '$\\Delta$ PIP RMSE vs exact', FKL: '$\\Delta$ Mean marginal KL vs exact', INTERVAL: '$\\Delta$ 95\\% interval score'}
    fig, axes = plt.subplots(1, 3, figsize=(16.2, 5.6))
    axes = np.asarray(axes).ravel()
    stage_offsets, stage_markers, stage_colors = _style_maps(STAGES)
    for i, (ax, metric) in enumerate(zip(axes, PLOT_METRICS)):
        dd = paired[paired['metric'].astype(str) == metric].copy()
        _draw_panel(ax, dd, metric, METHODS, STAGES, stage_offsets, stage_markers, stage_colors, PANEL_TITLES[metric])
        _panel_heading(ax, chr(97 + i), PANEL_TITLES[metric])
        if i > 0:
            ax.tick_params(axis='y', labelleft=False)
    _finish_legend(fig, axes, STAGES, y=0.03, ncol=len(STAGES))
    fig.suptitle('VIMCO-based methods versus ELBO across processing stages', fontsize=15, y=0.985)
    fig.subplots_adjust(left=0.09, right=0.99, top=0.8, bottom=0.18, wspace=0.38)
    out_dir = out / 'PAIRED_PLOTS_ITERATIVE_DAMPED_RELABELED'
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / 'figure_MAIN_BODYFAT_1x3_all_stages_paired_vs_ELBO_CLEAN_NO_OVERLAP.png', dpi=300, bbox_inches='tight', pad_inches=0.12)
    fig.savefig(out_dir / 'figure_MAIN_BODYFAT_1x3_all_stages_paired_vs_ELBO_CLEAN_NO_OVERLAP.pdf', bbox_inches='tight', pad_inches=0.12)
    paired.to_csv(out_dir / 'paired_comparisons_vs_ELBO_iterative_damped.csv', index=False)
    plt.close(fig)
    print('\nSaved:')
    print(' ', out_dir)

def plot_paired_v20_2x2(paired_clean, out):
    FIGURE_TITLE = 'VIMCO-based methods versus ELBO: Body Fat'
    PLOT_METRICS = ['PIP RMSE vs exact', 'Mean excess marginal FKL vs exact', 'Pareto k-hat', 'Exact-posterior expected 95% interval score']
    PANEL_TITLES = {'PIP RMSE vs exact': '$\\Delta$ PIP RMSE vs exact', 'Mean excess marginal FKL vs exact': '$\\Delta$ Mean marginal KL gap vs exact', 'Pareto k-hat': '$\\Delta$ Pareto $\\hat{k}$', 'Exact-posterior expected 95% interval score': '$\\Delta$ Expected interval score'}
    OUTPUT_STEM = 'figure_MAIN_bodyfat_2x2_all_stages_paired_vs_ELBO_CLEAN'
    METHOD_ORDER = ['VIMCO*', 'VIMCO* annealed', 'VIMCO-GM', 'VIMCO-GM annealed']
    STAGES = ['Original VI', 'Original PSIS', 'Damped PSIS', 'IWMM PSIS']
    STAGE_OFFSETS = {'Original VI': -0.27, 'Original PSIS': -0.09, 'Damped PSIS': 0.09, 'IWMM PSIS': 0.27}
    STAGE_MARKERS = {'Original VI': 'o', 'Original PSIS': 's', 'Damped PSIS': 'D', 'IWMM PSIS': '^'}
    default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    STAGE_COLORS = {stage: default_colors[i] for i, stage in enumerate(STAGES)}
    fig, axes = plt.subplots(2, 2, figsize=(14.4, 9.4))
    axes = axes.ravel()
    base_y = np.arange(len(METHOD_ORDER), dtype=float)
    method_index = {m: i for i, m in enumerate(METHOD_ORDER)}
    for panel_idx, (ax, metric) in enumerate(zip(axes, PLOT_METRICS)):
        dd = paired_clean[paired_clean['metric'].astype(str) == metric].copy()
        for stage in STAGES:
            ds = dd[dd['stage'].astype(str) == stage].copy()
            if ds.empty:
                continue
            ds['_order'] = ds['method'].map(method_index)
            ds = ds.sort_values('_order').set_index('method').reindex(METHOD_ORDER)
            x = pd.to_numeric(ds['oriented_mean_difference_left_is_better'], errors='coerce').to_numpy(float)
            lo = pd.to_numeric(ds['oriented_95% interval lower'], errors='coerce').to_numpy(float)
            hi = pd.to_numeric(ds['oriented_95% interval upper'], errors='coerce').to_numpy(float)
            y = base_y + STAGE_OFFSETS[stage]
            ok = np.isfinite(x) & np.isfinite(lo) & np.isfinite(hi)
            if np.any(ok):
                ax.errorbar(x[ok], y[ok], xerr=[x[ok] - lo[ok], hi[ok] - x[ok]], fmt=STAGE_MARKERS[stage], color=STAGE_COLORS[stage], capsize=3.2, elinewidth=1.15, linewidth=0, markersize=5.8, label=stage)
            points_only = np.isfinite(x) & ~ok
            if np.any(points_only):
                ax.plot(x[points_only], y[points_only], linestyle="none",
                    marker=STAGE_MARKERS[stage], color=STAGE_COLORS[stage], markersize=5.8,
                    label=stage if not np.any(ok) else "_nolegend_")
        ax.axvline(0.0, linestyle='--', linewidth=1.05, color='0.25', zorder=0)
        ax.set_yticks(base_y)
        ax.set_yticklabels(METHOD_ORDER)
        ax.invert_yaxis()
        ax.set_title(f'({chr(97 + panel_idx)}) {PANEL_TITLES[metric]}', loc='left', pad=10)
        ax.grid(axis='x', alpha=0.2)
        ax.set_xlabel('Paired difference vs ELBO')
        finite_bounds = []
        for col in ['oriented_95% interval lower', 'oriented_95% interval upper']:
            vals = pd.to_numeric(dd[col], errors='coerce').dropna()
            finite_bounds.extend(vals.tolist())
        if finite_bounds:
            b = float(np.max(np.abs(np.asarray(finite_bounds, dtype=float))))
            b = 1.12 * b if b > 0 else 1.0
            ax.set_xlim(-b, b)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
    handle_map = {}
    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()
        for handle, label in zip(handles, labels):
            if label not in handle_map:
                handle_map[label] = handle
    missing_legend = [s for s in STAGES if s not in handle_map]
    if missing_legend:
        raise RuntimeError('No plotted values were found for these stages: ' + ', '.join(missing_legend))
    fig.legend([handle_map[s] for s in STAGES], STAGES, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.018))
    fig.suptitle(FIGURE_TITLE, fontsize=15, y=0.975)
    fig.subplots_adjust(left=0.115, right=0.975, top=0.895, bottom=0.135, wspace=0.31, hspace=0.34)
    PLOT_DIR_CLEAN = out / 'plots'
    PLOT_DIR_CLEAN.mkdir(parents=True, exist_ok=True)
    png_path = PLOT_DIR_CLEAN / f'{OUTPUT_STEM}.png'
    pdf_path = PLOT_DIR_CLEAN / f'{OUTPUT_STEM}.pdf'
    fig.savefig(png_path, dpi=300, bbox_inches='tight', pad_inches=0.12)
    fig.savefig(pdf_path, bbox_inches='tight', pad_inches=0.12)
    plt.close(fig)
    print('\nSaved clean figure:')
    print(' ', png_path)
    print(' ', pdf_path)

def digest_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def digest_object(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()

def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=False, float_format='%.17g')
    tmp.replace(path)

def summary_table(data, keys):
    rows = []
    for group, g in data.groupby(keys, sort=False, dropna=False):
        group = group if isinstance(group, tuple) else (group,)
        row = dict(zip(keys, group))
        for m in METRICS:
            x = g[m].to_numpy(float)
            finite = x[np.isfinite(x)]
            row[m] = float(np.mean(x))
            row[m + ' SD across optimisation seeds'] = float(np.std(x, ddof=1)) if len(x) > 1 and len(finite) == len(x) else np.nan
            row[m + ' n_finite'] = len(finite)
        row['n_optimisation seeds'] = len(g)
        rows.append(row)
    return pd.DataFrame(rows)

def export_table(df, stem, out, caption):
    atomic_csv(df, out / f'{stem}.csv')
    view = df.copy()
    for metric in METRICS:
        sdcol = metric + ' SD across optimisation seeds'
        if sdcol in view:
            view[metric] = [f'{m:.4f} ({sd:.4f})' if np.isfinite(sd) else f'{m:.4f} (--)' for m, sd in zip(view[metric], view[sdcol])]
            view = view.drop(columns=[sdcol, metric + ' n_finite'], errors='ignore')
    view = view.rename(columns={'PIP RMSE vs exact': 'RMSE', 'Mean excess marginal FKL vs exact': 'Mean excess FKL', 'Pareto k-hat': 'k-hat', 'Exact posterior mass in 95% latent interval': 'Exact interval mass', 'Exact-posterior expected 95% interval score': 'Expected interval score'})
    tex = view.to_latex(index=False, escape=True, float_format=lambda x: f'{x:.4f}', na_rep='--', caption=caption, label='tab:' + stem.lower(), longtable=len(df) > 35)
    (out / f'{stem}.tex').write_text(tex, encoding='utf-8')

def paired_statistics(records):
    rows = []
    for stage in STAGES:
        e = records[(records.method == 'ELBO') & (records.stage == stage)].set_index('seed')
        for method in METHODS[1:]:
            d = records[(records.method == method) & (records.stage == stage)].set_index('seed')
            if set(d.index) != set(e.index) or d.index.duplicated().any() or e.index.duplicated().any():
                raise ValueError('Paired comparisons require exactly the same optimisation seeds.')
            for metric in METRICS:
                raw = d[metric] - e[metric]
                oriented = (d[metric] - 0.95).abs() - (e[metric] - 0.95).abs() if metric == 'Exact posterior mass in 95% latent interval' else raw
                for_plot = _ci95(oriented.replace([np.inf, -np.inf], np.nan))
                sr = _ci95(raw.replace([np.inf, -np.inf], np.nan))
                rows.append(dict(stage=stage, method=method, metric=metric, n=for_plot['n'], n_pairs_total=len(raw), mean_paired_difference=sr['mean'], sd_paired_difference=sr['sd'], oriented_mean_difference_left_is_better=for_plot['mean'], **{'oriented_95% interval lower': for_plot['lo'], 'oriented_95% interval upper': for_plot['hi'], '95% interval lower': sr['lo'], '95% interval upper': sr['hi']}))
    return pd.DataFrame(rows)

def sensitivity_tables(records, candidates, selected, out):
    keys = ['method', 'stage', 'seed']
    lr = records.groupby(keys + ['learning_rate'], sort=False)[METRICS].mean().reset_index()
    t1 = summary_table(lr, ['method', 'stage', 'learning_rate'])
    export_table(t1, 'SENSITIVITY_1_learning_rate', out, 'Learning-rate sensitivity: average over alpha and available S,N allocations within each optimisation seed, then average across optimisation seeds. SD describes optimisation seed variation, not configuration variation.')
    allocation = records.groupby(keys + ['S', 'N'], sort=False)[METRICS].mean().reset_index()
    t2 = summary_table(allocation, ['method', 'stage', 'S', 'N'])
    t2.insert(0, 'panel', 'Allocation average over learning rate and alpha')
    t2['policy'] = 'All configurations in allocation'
    t2['comparison_status'] = 'S sensitivity unavailable: only one allocation per method was fitted'
    export_table(t2, 'SENSITIVITY_2_S_N', out, 'Observed allocation averages only: VIMCO S=1,N=300 and ELBO S=300,N=1. No evidence about varying S is available.')
    fixed = records[records.method.eq('ELBO') | (records.S == 1) & (records.N == 300)].copy()
    per_seed = fixed.groupby(keys, sort=False)[METRICS].mean().reset_index()
    t3 = summary_table(per_seed, ['method', 'stage'])
    counts = fixed.groupby(['method', 'stage', 'seed']).size().groupby(['method', 'stage']).first().rename('configurations_per_optimisation seed').reset_index()
    t3 = t3.merge(counts, on=['method', 'stage'], validate='one_to_one')
    export_table(t3, 'SENSITIVITY_3_mean_across_configurations_S1', out, 'S=1 configuration-averaged performance (ELBO S=300,N=1): average alpha/learning-rate configurations within each optimisation seed, then average optimisation seeds. This table does not select the best configuration.')
    rows = []
    for (method, stage), g in fixed.groupby(['method', 'stage'], sort=False):
        by_setting = g.groupby('setting')[METRICS].mean()
        within_seed = g.groupby('seed')[METRICS].std(ddof=1)
        for metric in METRICS:
            vals = by_setting[metric].to_numpy(float)
            w = within_seed[metric].to_numpy(float)
            rows.append({'method': method, 'stage': stage, 'metric': metric, 'n_configurations': len(by_setting), 'SD across configuration means': float(np.std(vals, ddof=1)) if len(vals) > 1 and np.isfinite(vals).all() else np.nan, 'Mean within-optimisation seed configuration SD': float(np.mean(w)) if np.isfinite(w).all() else np.nan, 'n_optimisation seeds': g.seed.nunique()})
    t4 = pd.DataFrame(rows)
    export_table(t4, 'SENSITIVITY_4_SD_across_configurations_S1', out, 'Configuration sensitivity at S=1: sample SD across configuration means (each mean is over identical optimisation seeds), and average within-optimisation seed sample SD across configurations. Undefined SDs remain missing. Small SD alone does not establish good performance.')



## Data and saved training outputs


In [ ]:
def load_bodyfat():
    candidates = [ROOT / 'bodyfat.csv', ROOT / 'bodyfat' / 'bodyfat.csv',
                  ROOT.parent / 'bodyfat' / 'bodyfat.csv']
    path = next((p for p in candidates if p.is_file()), None)
    if path is None:
        raise FileNotFoundError('Cannot find bodyfat.csv: ' + ', '.join(map(str, candidates)))
    frame = pd.read_csv(path)
    response = next((c for c in ['y', 'body_fat', 'brozek_fat', 'bodyfat',
                                'percent_body_fat'] if c in frame), None)
    if response is None:
        raise ValueError('No recognised Body Fat response column.')
    names = [c for c in frame if c != response and c != 'case_id'
             and not str(c).startswith('Unnamed')]
    if len(names) != P:
        raise ValueError(f'Expected {P} predictors; found {len(names)}.')
    X = np.ascontiguousarray(frame[names].to_numpy(np.float64))
    y = np.ascontiguousarray(frame[response].to_numpy(np.float64))
    if len(y) < 3 or not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError('Invalid Body Fat observations.')
    means, sd = X.mean(0), X.std(0, ddof=0)
    sd[sd == 0.0] = 1.0
    X -= means
    X /= sd
    return X, y, names, path


def make_target(X, y):
    return CollapsedLinearTarget(X, y, TargetSpec(TAU2, A_PI, B_PI, jitter=1e-8,
        primal_active_max=50, primal_batch_size=64, dual_batch_size=8))


@lru_cache(maxsize=300)
def _wide(path):
    return pd.read_csv(path)


def read_q(row, names):
    wide = _wide(str(row['pip_file']))
    if (len(wide) != P or 'variable_name' not in wide or
            wide.variable_name.astype(str).tolist() != list(map(str, names))):
        raise ValueError(f'Predictor order or dimension mismatch: {row["pip_file"]}')
    pip = wide[str(row['pip_column'])].to_numpy(np.float64)
    eta = wide[str(row['lambda_column'])].to_numpy(np.float64)
    if (not np.isfinite(pip).all() or not np.isfinite(eta).all() or
            np.any((pip < 0) | (pip > 1)) or
            not np.allclose(pip, expit(eta), atol=1e-10, rtol=0)):
        raise ValueError(f'Invalid saved PIPs/logits: {row["pip_file"]}')
    return pip.copy(), eta.copy()


def _training_output_files(folder, stem):
    pairs = [('final', folder / f'pips_lambda_{stem}',
              folder / f'training_summary_{stem}'),
             ('original', folder / f'pips_{stem}',
              folder / f'runtime_and_metrics_{stem}')]
    for output_format, pip_path, summary_path in pairs:
        if pip_path.is_file() and summary_path.is_file():
            return pip_path, summary_path, output_format
    expected = '\n'.join(f'  {pip_path}\n  {summary_path}'
                         for _, pip_path, summary_path in pairs)
    raise FileNotFoundError('Missing training outputs. Expected either complete pair:\n' + expected)


def _training_summary(summary_path, output_format, allocation):
    summary = pd.read_csv(summary_path)
    if output_format == 'final':
        required = {'method', 'split_label', 'setting', 'runtime_seconds', 'iterations',
                    'final_alpha', 'tau2', 'a_pi', 'b_pi', 'method_seed', 'learning_rate',
                    'alpha_setting', 'S', 'N', 'train_file_sha256'}
    else:
        required = {'method', 'method_seed_run', 'method_seed', 'pip_column', 'lambda_column',
                    'S', 'N', 'learning_rate', 'alpha_setting', 'final_alpha', 'iters',
                    'training_time_seconds', 'a_pi', 'b_pi', 'prior_expected_model_size'}
    if not required.issubset(summary):
        missing = ', '.join(sorted(required.difference(summary.columns)))
        raise ValueError(f'Missing training metadata ({missing}) in {summary_path}')
    if output_format == 'original':
        if not summary.method_seed.astype(float).eq(summary.method_seed_run.astype(float)).all():
            raise ValueError(f'Conflicting method seeds in {summary_path}')
        if not np.allclose(summary.prior_expected_model_size.astype(float), M0,
                           atol=1e-14, rtol=0):
            raise ValueError(f'Incompatible prior_expected_model_size in {summary_path}')
        settings = summary.pip_column.astype(str).str.removeprefix('pip_')
        if (not summary.pip_column.astype(str).eq('pip_' + settings).all() or
                not summary.lambda_column.astype(str).eq('lambda_' + settings).all()):
            raise ValueError(f'Invalid PIP/logit column mapping in {summary_path}')
        summary = summary.assign(setting=settings, split_label=allocation,
            runtime_seconds=summary.training_time_seconds, iterations=summary.iters,
            method_seed=summary.method_seed_run)
    return summary


def available_method_seeds():
    """Use the same available optimisation seeds for every method."""
    if METHOD_SEEDS is not None:
        requested = tuple(METHOD_SEEDS)
        if (not requested or len(set(requested)) != len(requested) or
                any(isinstance(seed, bool) or not isinstance(seed, (int, np.integer))
                    for seed in requested)):
            raise ValueError('METHOD_SEEDS must be None or distinct integer optimisation seeds.')
    else:
        requested = None
    complete, seen = {}, set()
    for method in METHODS:
        allocation = 'S_300_N_1' if method == 'ELBO' else 'S_1_N_300'
        token = method_token(method)
        folder = TRAINING_ROOT / f'output_{token}_bodyfat_cython'
        seeds = set()
        for prefix in (f'pips_lambda_{token}_{allocation}_methodseed_',
                       f'pips_{token}_{allocation}_methodseed_'):
            for path in folder.glob(prefix + '*.csv'):
                suffix = path.name[len(prefix):-4]
                if not suffix.isdecimal():
                    continue
                seed = int(suffix)
                if suffix != str(seed) or not path.is_file():
                    continue
                seen.add(seed)
                stem = f'{token}_{allocation}_methodseed_{seed}.csv'
                try:
                    _training_output_files(folder, stem)
                except FileNotFoundError:
                    continue
                seeds.add(seed)
        complete[method] = seeds
    shared = set.intersection(*complete.values()) if complete else set()
    selected = tuple(sorted(shared)) if requested is None else tuple(
        int(seed) for seed in requested if seed in shared)
    counts = ', '.join(f'{method}: {len(seeds)}' for method, seeds in complete.items())
    print('Seeds with complete file pairs by method: ' + counts, flush=True)
    incomplete = sorted(seen - shared)
    if incomplete:
        print('Skipping seeds without complete file pairs for every method: ' +
              ', '.join(map(str, incomplete)), flush=True)
    if requested is not None:
        missing = [int(seed) for seed in requested if seed not in shared]
        if missing:
            print('Requested seeds unavailable across all methods: ' +
                  ', '.join(map(str, missing)), flush=True)
    if not selected:
        raise FileNotFoundError('No common available method seeds in ' + str(TRAINING_ROOT) +
            '. Each selected seed needs a PIP/logit CSV and its matching summary CSV for every '
            'method, at ELBO S=300,N=1 or VIMCO S=1,N=300.')
    print(f'Post-processing {len(selected)} shared method seeds: ' +
          ', '.join(map(str, selected)), flush=True)
    return selected


def load_candidates(names, data_path):
    method_seeds = available_method_seeds()
    train_hash = digest_file(data_path)
    rows = []
    formats = set()
    for seed in method_seeds:
        for method in METHODS:
            allocation = 'S_300_N_1' if method == 'ELBO' else 'S_1_N_300'
            stem = f'{method_token(method)}_{allocation}_methodseed_{seed}.csv'
            folder = TRAINING_ROOT / f'output_{method_token(method)}_bodyfat_cython'
            pip_path, summary_path, output_format = _training_output_files(folder, stem)
            wide = _wide(str(pip_path))
            summary = _training_summary(summary_path, output_format, allocation)
            formats.add(output_format)
            grid = expected_grid(method)
            settings = [v[4] for v in grid]
            columns = ['variable_name']
            for setting in settings:
                columns.extend([f'pip_{setting}', f'lambda_{setting}'])
            if (wide.columns.tolist() != columns or len(wide) != P or
                    wide.variable_name.astype(str).tolist() != list(map(str, names)) or
                    len(summary) != len(grid) or summary.setting.astype(str).tolist() != settings or
                    not summary.method.eq(method).all() or
                    not summary.split_label.eq(allocation).all()):
                raise ValueError(f'Incomplete method grid or predictor order in {pip_path}')
            if not summary.method_seed.astype(float).eq(float(seed)).all():
                raise ValueError(f'Method seed mismatch in {summary_path}')
            hash_verified = 'train_file_sha256' in summary
            tau2_recorded = 'tau2' in summary
            if hash_verified and not summary.train_file_sha256.astype(str).eq(train_hash).all():
                raise ValueError(f'Input-data hash mismatch in {summary_path}')
            priors = [('a_pi', A_PI), ('b_pi', B_PI)]
            if tau2_recorded:
                priors.append(('tau2', TAU2))
            for column, value in priors:
                if not np.allclose(summary[column].astype(float), value, atol=1e-14, rtol=0):
                    raise ValueError(f'Incompatible {column} in {summary_path}')
            pip_hash, summary_hash = digest_file(pip_path), digest_file(summary_path)
            for i, (lr, alpha, S, N, setting) in enumerate(grid):
                record = summary.iloc[i]
                runtime, iterations = float(record.runtime_seconds), int(record.iterations)
                if (not np.isfinite(runtime) or runtime < 0 or not 1 <= iterations <= 3000 or
                        not np.isclose(float(record.learning_rate), lr, atol=1e-14, rtol=0) or
                        int(record.S) != S or int(record.N) != N):
                    raise ValueError(f'Invalid setting/runtime/iterations: {summary_path} {setting}')
                if isinstance(alpha, (int, float)) and not np.isclose(float(record.alpha_setting), alpha):
                    raise ValueError(f'Wrong alpha setting: {summary_path} {setting}')
                row = dict(method=method, seed=int(seed), method_seed=int(seed),
                    tau2=TAU2, a_pi=A_PI, b_pi=B_PI, S=S, N=N, learning_rate=lr,
                    alpha_setting=alpha, setting=setting, iterations=iterations,
                    final_alpha=record.final_alpha, training_runtime_seconds=runtime,
                    pip_file=str(pip_path), pip_column=f'pip_{setting}',
                    lambda_column=f'lambda_{setting}', training_summary_file=str(summary_path),
                    training_output_format=output_format, training_tau2_recorded=tau2_recorded,
                    training_data_hash_verified=hash_verified,
                    train_sha256=train_hash, pip_sha256=pip_hash, summary_sha256=summary_hash)
                _, eta = read_q(row, names)
                row['proposal_hash'] = hashlib.sha256(eta.tobytes()).hexdigest()
                row['config_id'] = digest_object([ANALYSIS_CODE_SHA256, method, seed, setting,
                    train_hash, pip_hash, summary_hash, row['proposal_hash']])[:32]
                rows.append(row)
        print(f'Validated method seed {seed}: five complete method grids.', flush=True)
    if 'original' in formats:
        print('Read original pips_/runtime_and_metrics_ CSVs. Missing training tau2/data-hash '
              'metadata is marked as unrecorded; analysis uses tau2=1 and the current bodyfat.csv.',
              flush=True)
    cand = pd.DataFrame(rows)
    if cand.duplicated(['method', 'seed', 'setting']).any() or cand.config_id.duplicated().any():
        raise ValueError('Duplicate configuration identity.')
    return cand


def evaluate_candidates(cand, out, target, names):
    rows = []
    for i, row in cand.iterrows():
        _, eta = read_q(row, names)
        ev = {**_eval_elbo_2000(target, eta, EVAL_BATCH_SIZE),
              'evaluated_elbo_2000 source': 'new_common_seed_evaluation'}
        if _evaluation_fields(ev) is None:
            raise ValueError(f'Invalid evaluated ELBO for {row.config_id}')
        rows.append({**row.to_dict(), **ev})
        if (i + 1) % 5 == 0 or i + 1 == len(cand):
            print(f'ELBO evaluated {i+1}/{len(cand)} configurations', flush=True)
    evaluated = pd.DataFrame(rows)
    atomic_csv(evaluated, out / 'evaluated_configurations.csv')
    return evaluated


def choose_configurations(evaluated):
    chosen = pd.DataFrame([_select_best_mcse(g) for _, g in
                           evaluated.groupby(['method', 'seed'], sort=False)])
    seeds = evaluated.seed.astype(int).unique()
    required = {(method, int(seed)) for method in METHODS for seed in seeds}
    found = set(zip(chosen.method, chosen.seed.astype(int)))
    if found != required or chosen.duplicated(['method', 'seed']).any():
        raise ValueError('Expected one selected fit per method and optimisation seed.')
    return chosen.reset_index(drop=True)




## Exact reference and configuration scoring


In [ ]:
def exact_reference(out, X, y, names, target):
    started = time.perf_counter()
    count = 1 << P
    ids = np.arange(count, dtype=np.uint32)[:, None]
    G = ((ids >> np.arange(P, dtype=np.uint32)[None, :]) & 1).astype(np.uint8)
    values = []
    with torch.no_grad():
        for start in range(0, count, 512):
            gt = torch.from_numpy(G[start:start+512].astype(np.float64)).to(dtype=DTYPE, device=DEVICE)
            values.append(target.log_joint(gt).cpu().numpy())
    logjoint = np.concatenate(values)
    if not np.isfinite(logjoint).all():
        raise FloatingPointError('Non-finite exact model log posterior.')
    log_evidence = float(logsumexp(logjoint))
    weights = np.exp(logjoint - log_evidence)
    weights /= weights.sum()
    pip = weights @ G
    EXACT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    exact_path = EXACT_OUTPUT_DIR / 'exact_model_posterior_primal_m0_3_tau2_1_final.csv'
    models = pd.DataFrame(G, columns=[f'gamma_{name}' for name in names])
    models['log_joint'] = logjoint
    models['posterior_probability'] = weights
    atomic_csv(models, exact_path)
    pip_path = EXACT_OUTPUT_DIR / 'exact_pips_primal_m0_3_tau2_1_final.csv'
    atomic_csv(pd.DataFrame({'variable_name': names, 'exact_pip': pip}), pip_path)
    Xstar = X[:min(N_XSTAR, len(X))]
    loc, scale2, nu = _posterior_components(G, X, y, Xstar, tau2=TAU2, observation_noise=False)
    rng = np.random.default_rng(663001)
    idx = rng.choice(count, size=N_EXACT_REFERENCE_DRAWS, replace=True, p=weights)
    draws = loc[idx] + np.sqrt(scale2[idx]) * rng.standard_t(
        df=nu, size=(N_EXACT_REFERENCE_DRAWS, len(Xstar)))
    info = dict(source=str(exact_path), log_evidence=log_evidence, tau2=TAU2,
                a_pi=A_PI, b_pi=B_PI, n_models=count, data_sha256=digest_file(load_bodyfat()[3]),
                runtime_seconds=time.perf_counter()-started)
    (EXACT_OUTPUT_DIR / 'exact_reference_tau2_1_final.json').write_text(json.dumps(info, indent=2))
    atomic_csv(pd.DataFrame({'variable_name': names, 'exact_pip': pip}), out / 'exact_reference_pips.csv')
    return dict(pip=pip, draws=draws, source=str(exact_path), metadata=info)


def score_configuration(row, reference, X, y, names, target):
    pip_vi, eta = read_q(row, names)
    seed, method, label = int(row.seed), str(row.method), str(row.pip_column)
    start = time.perf_counter()
    ordinary = draw_and_score(target, eta, stable_seed(seed, method, label, '|ORDINARY_SHARED'),
                              batch_size=BATCH_SIZE)
    ordinary_time = time.perf_counter() - start
    start = time.perf_counter()
    damp = run_damped(target, eta, method, seed, label, ADAPTED_PROB_EPS, BATCH_SIZE,
                      ordinary_override=ordinary)
    damp_time = time.perf_counter() - start
    start = time.perf_counter()
    iwmm = run_iwmm(target, eta, method, seed, label, ADAPTED_PROB_EPS, BATCH_SIZE,
                    ordinary_override=ordinary)
    iwmm_time = time.perf_counter() - start
    stages = {
        'Original VI': (pip_vi, ordinary.G, np.full(K_FIXED, 1.0/K_FIXED), ordinary.khat, 0.0),
        'Original PSIS': (ordinary.pip, ordinary.G, ordinary.psis_weights, ordinary.khat, ordinary_time),
        'Damped PSIS': (damp['final'].pip, damp['final'].G, damp['final'].psis_weights,
                        damp['final'].khat, ordinary_time+damp_time),
        'IWMM PSIS': (iwmm['final'].pip, iwmm['final'].G, iwmm['final'].psis_weights,
                      iwmm['final'].khat, ordinary_time+iwmm_time)}
    records, pips, components = [], {}, {}
    Xstar = X[:min(N_XSTAR, len(X))]
    for stage, (pip, G, weights, khat, post_time) in stages.items():
        if id(G) not in components:
            components[id(G)] = _posterior_components(G, X, y, Xstar, tau2=TAU2,
                                                       observation_noise=False)
        loc, scale2, nu = components[id(G)]
        _, lo, hi = _interval_metrics(loc, scale2, weights, nu, 770001+seed, truth=None)
        draws = reference['draws']
        mass = float(np.mean((draws >= lo) & (draws <= hi)))
        score = float(np.mean((hi-lo) + 40*np.maximum(lo-draws, 0) + 40*np.maximum(draws-hi, 0)))
        record = {k: row[k] for k in ['config_id', 'method', 'seed', 'method_seed', 'setting',
                                     'learning_rate', 'alpha_setting', 'S', 'N', 'tau2']}
        record.update({'stage': stage, 'PIP RMSE vs exact': _rmse(pip, reference['pip']),
            'Mean excess marginal FKL vs exact': forward_kl_product_bernoulli(reference['pip'], pip)/P,
            'Pareto k-hat': float(khat), 'Exact posterior mass in 95% latent interval': mass,
            'Exact-posterior expected 95% interval score': score,
            'Training runtime (s)': float(row.training_runtime_seconds),
            'Post-processing runtime (s)': float(post_time),
            'Total runtime (s)': float(row.training_runtime_seconds+post_time),
            'Selected VI evaluated ELBO': float(row.evaluated_elbo_2000),
            'importance_ESS': float(1.0 / np.sum(np.asarray(weights, float) ** 2)) if stage != 'Original VI' else np.nan})
        records.append(record)
        pips[stage] = np.asarray(pip, np.float64)
    audit = dict(config_id=row.config_id, method=method, method_seed=seed,
        ordinary_seed=ordinary.seed, damped_final_seed=damp['final'].seed,
        iwmm_final_seed=iwmm['final'].seed, ordinary_k=float(ordinary.khat),
        damped_k=float(damp['final'].khat), iwmm_k=float(iwmm['final'].khat),
        damped_triggered=bool(damp['triggered']), iwmm_triggered=bool(iwmm['triggered']),
        damped_accepted_iterations=int(damp['accepted_iterations']),
        damped_accepted_omegas=json.dumps(damp['accepted_omegas']),
        damped_stop_reason=damp['stop_reason'], iwmm_accepted_iterations=int(iwmm['accepted_iterations']),
        iwmm_stop_reason=iwmm['stop_reason'])
    history = [{**h, 'config_id': row.config_id, 'method': method, 'method_seed': seed,
                'adaptation': kind} for kind, result in [('Damped', damp), ('IWMM', iwmm)]
               for h in result['history']]
    return dict(records=records, pips=pips, audit=audit, history=history)




## Run post-processing


In [ ]:
def run_postprocessing():
    plt.rcParams['text.usetex'] = False
    torch.set_num_threads(TORCH_THREADS)
    validate_psis_backend_sign()
    _wide.cache_clear()
    X, y, names, data_path = load_bodyfat()
    target = make_target(X, y)
    cand = load_candidates(names, data_path)
    versions = dict(torch=str(torch.__version__), numpy=np.__version__, pandas=pd.__version__,
                    scipy=__import__('scipy').__version__, arviz=str(getattr(az, '__version__', None)),
                    python=platform.python_version())
    signature = digest_object(dict(code=ANALYSIS_CODE_SHA256, data=digest_file(data_path),
        proposals=cand[['config_id', 'proposal_hash']].to_dict('records'), K=K_FIXED,
        psis_base_seed=PSIS_BASE_SEED, eval_seed=EVAL_ELBO_SEED, eval_n=EVAL_ELBO_SAMPLES,
        exact_draws=N_EXACT_REFERENCE_DRAWS, mixture_draws=N_MIXTURE_DRAWS, versions=versions))
    out = OUTPUT_DIR / signature[:16]
    out.mkdir(parents=True, exist_ok=True)
    atomic_csv(cand, out / 'candidate_configurations.csv')
    evaluated = evaluate_candidates(cand, out, target, names)
    selected = choose_configurations(evaluated)
    selected_ids = set(selected.config_id)
    atomic_csv(selected, out / 'selected_settings_S1_N300.csv')
    reference = exact_reference(out, X, y, names, target)
    pip_dir = out / 'corrected_pips'
    pip_dir.mkdir(exist_ok=True)
    records, audits, history, scatter = [], [], [], []
    for i, row in evaluated.iterrows():
        result = score_configuration(row, reference, X, y, names, target)
        records.extend(result['records'])
        audits.append(result['audit'])
        history.extend(result['history'])
        if row.config_id in selected_ids:
            wide = pd.DataFrame({'variable_name': names, 'Exact posterior': reference['pip'],
                                 **result['pips']})
            atomic_csv(wide, pip_dir / f'{method_token(str(row.method))}_methodseed_{int(row.seed)}.csv')
            for stage, pip in result['pips'].items():
                scatter.extend(dict(seed=int(row.seed), method=row.method, stage=stage,
                    variable_name=name, reference_pip=float(reference['pip'][j]), method_pip=float(pip[j]))
                    for j, name in enumerate(names))
        if (i+1) % 5 == 0 or i+1 == len(evaluated):
            print(f'Post-processing {i+1}/{len(evaluated)}: {row.method}, method seed {int(row.seed)}', flush=True)
    records = pd.DataFrame(records)
    main = records[records.config_id.isin(selected_ids)].copy()
    atomic_csv(records, out / 'all_configuration_metrics.csv')
    atomic_csv(main, out / 'selected_configuration_metrics.csv')
    atomic_csv(pd.DataFrame(audits), out / 'adaptation_audit.csv')
    atomic_csv(pd.DataFrame(history), out / 'adaptation_history.csv')
    paper = summary_table(main, ['method', 'stage'])
    export_table(paper, 'MAIN_selected_S1_metrics', out,
        'Body Fat: means and SD across optimisation seeds. Exact-posterior PIPs are the reference. '
        'Interval mass and expected interval score use exact-posterior draws for the latent mean.')
    atomic_csv(paper, out / 'main_metrics_numeric.csv')
    paired = paired_statistics(main)
    atomic_csv(paired, out / 'paired_comparisons_vs_ELBO.csv')
    plot_paired_v20(paired, out)
    plot_paired_v20_2x2(paired, out)
    scatter = pd.DataFrame(scatter)
    atomic_csv(scatter, out / 'selected_pips_all_stages.csv')
    make_reference_scatter_grid(scatter[scatter.stage.eq('Original VI')], out, 'bodyfat')
    make_stage_scatter_2x2(scatter, out, 'bodyfat')
    sensitivity_tables(records, evaluated, selected, out)
    atomic_csv(selected[['method', 'method_seed', 'tau2', 'learning_rate', 'alpha_setting',
                        'iterations', 'final_alpha', 'training_runtime_seconds',
                        'evaluated_elbo_2000', 'evaluated_elbo_2000 MCSE']],
               out / 'selected_training_alpha_iterations_runtime.csv')
    manifest = dict(code_sha256=ANALYSIS_CODE_SHA256, run_signature=signature,
        tau2=TAU2, m0=M0, a_pi=A_PI, b_pi=B_PI, data_path=str(data_path),
        data_sha256=digest_file(data_path), method_seeds=sorted(cand.seed.astype(int).unique().tolist()), workers=WORKERS,
        seed_selection='all_shared_available' if METHOD_SEEDS is None else 'requested_available_subset',
        requested_method_seeds=None if METHOD_SEEDS is None else list(METHOD_SEEDS),
        psis_seed_rule='1217 + method_seed + 100000*stage + 1000*(iteration-1)',
        psis_stages={'ordinary': 0, 'damped_pilot': 1, 'damped_validation': 2,
                     'iwmm_candidate': 3, 'final': 4},
        selection_rule='V20: 2 combined MCSE ties, then smallest MCSE',
        allocation={'ELBO': [300, 1], 'VIMCO methods': [1, 300]},
        exact_reference=reference['metadata'], n_configurations=len(cand), n_selected=len(selected),
        training_output_formats=cand.training_output_format.value_counts().to_dict(),
        training_metadata=dict(tau2_recorded_for_all=bool(cand.training_tau2_recorded.all()),
            input_data_hash_verified_for_all=bool(cand.training_data_hash_verified.all())),
        K=K_FIXED, versions=versions)
    (out / 'analysis_manifest.json').write_text(json.dumps(manifest, indent=2))
    (OUTPUT_DIR / 'LATEST_RESULTS.txt').write_text(str(out) + '\n')
    print('Saved all post-processing:', out)
    return dict(out=out, selected=selected, records=main, all_records=records,
                main_table=paper, paired=paired, reference=reference)


RESULTS = run_postprocessing()


## Thesis tables and figures

Built only from the saved post-processing results above; nothing is retrained or re-weighted. Outputs go to `<results folder>/THESIS_TABLES_AND_FIGURES`: the main results table with Pareto $\hat k$ and ESS (`.csv` and `.tex`), selected $\alpha$, iterations and runtime, the learning-rate summary, the Original VI configuration-sensitivity summary, and the paired and PIP figures.

In [ ]:
"""Thesis tables and figures, built only from the saved post-processing results.

make_thesis_outputs(setting, results_dir) writes everything into
results_dir / 'THESIS_TABLES_AND_FIGURES'. Nothing is retrained or re-weighted here.

Main results table (Tables 4.3-4.5, 5.1, 5.2) with the Pareto k-hat and ESS columns
Selected fixed alpha and final annealed alpha (Table 5.4)
Mean iterations to termination and selected-fit runtime (Tables 5.5, 5.3)
Learning rate maximising configuration-averaged evaluated ELBO (Table C.1)
Original VI configuration averages and within-replicate configuration SD (Table G.1)
Paired comparisons with ELBO (Figures 4.3, 4.5, 4.6, 5.1, 5.2) and PIP scatter plots
p = 30 only: evaluated-ELBO allocation F-tests (Table B.1) and the varbvs benchmark row
"""
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FormatStrFormatter
from scipy.stats import f as f_distribution

TABLE_METHODS = ['ELBO', 'VIMCO*', 'VIMCO* annealed', 'VIMCO-GM', 'VIMCO-GM annealed']
METHOD_TEX = {'ELBO': 'ELBO', 'VIMCO*': r'\vstar', 'VIMCO* annealed': r'\avstar',
              'VIMCO-GM': r'\vgm', 'VIMCO-GM annealed': r'\avgm'}
TABLE_STAGES = ['Original VI', 'Original PSIS', 'IWMM PSIS', 'Damped PSIS']
STAGE_TEX = {'Original VI': 'Original VI', 'Original PSIS': 'Ordinary PSIS', 'IWMM PSIS': 'IWMM',
             'Damped PSIS': r'\shortstack[l]{Iterative Damped\\PSIS}'}
K_COL, ESS_COL = 'Pareto k-hat', 'importance_ESS'

# Columns of each results table: (metric column, LaTeX header, decimals).
# vi_k: the Original VI row shows the k-hat of the VI proposal (= Ordinary PSIS k-hat).
SETTINGS = {
    'p30': dict(
        label='p30', replicates='datasets', vi_k=False,
        cols=[('PIP RMSE vs Gibbs', 'PIP RMSE', 4), ('Mean excess marginal FKL vs Gibbs', r'$\Delta\mathrm{FKL}$', 4),
              (K_COL, r'$\widehat k$', 3), (ESS_COL, 'ESS', 0), ('95% latent coverage', 'Coverage', 4),
              ('95% interval score', 'Interval score', 4)],
        sensitivity=[('PIP RMSE vs Gibbs', 'PIP RMSE'), ('Mean excess marginal FKL vs Gibbs', 'Excess marginal KL'),
                     ('95% latent coverage', 'Coverage'), ('95% interval score', 'Interval score')],
        all_metrics='all_configuration_metrics_S1.csv', evaluated='evaluated_configurations.csv',
        selected_training='selected_settings_S1_N300.csv',
        paired=[('PIP RMSE vs Gibbs', 'Δ PIP RMSE vs Gibbs'), ('Mean excess marginal FKL vs Gibbs', 'Δ Mean marginal KL vs Gibbs'),
                ('95% interval score', 'Δ 95% interval score')]),
    'p1000': dict(
        label='p1000', replicates='datasets', vi_k=False,
        cols=[('PIP RMSE vs Gibbs', 'PIP RMSE', 4), ('Mean excess marginal FKL vs Gibbs', r'$\Delta\mathrm{FKL}$', 4),
              ('PIP RMSE vs truth', 'PIP RMSE', 4), ('F1 vs truth', '$F_1$', 4), (K_COL, r'$\widehat k$', 3),
              (ESS_COL, 'ESS', 0), ('95% latent coverage', 'Coverage', 4), ('95% interval score', 'Interval score', 4)],
        sensitivity=[('PIP RMSE vs Gibbs', 'PIP RMSE'), ('Mean excess marginal FKL vs Gibbs', 'Excess marginal KL'),
                     ('95% latent coverage', 'Coverage'), ('95% interval score', 'Interval score')],
        all_metrics='all_configuration_metrics.csv', evaluated='all_configurations_evaluated.csv',
        selected_training='selected_configurations.csv',
        paired=[('PIP RMSE vs Gibbs', 'Δ PIP RMSE vs Gibbs'), ('Mean excess marginal FKL vs Gibbs', 'Δ Mean marginal KL vs Gibbs'),
                ('PIP RMSE vs truth', 'Δ PIP RMSE vs truth'), ('F1 vs truth', '−Δ F1 vs truth'),
                ('95% interval score', 'Δ 95% interval score')]),
    'p5000': dict(
        label='p5000', replicates='datasets', vi_k=False,
        cols=[('PIP RMSE vs truth', 'PIP RMSE', 4), ('F1 vs truth', '$F_1$', 4), (K_COL, r'$\widehat k$', 3),
              (ESS_COL, 'ESS', 0), ('95% latent coverage', 'Coverage', 4), ('95% interval score', 'Interval score', 4)],
        sensitivity=[('PIP RMSE vs truth', 'PIP RMSE'), ('F1 vs truth', '$F_1$'),
                     ('95% latent coverage', 'Coverage'), ('95% interval score', 'Interval score')],
        all_metrics='all_configuration_metrics.csv', evaluated='all_configurations_evaluated.csv',
        selected_training='selected_configurations.csv',
        paired=[('PIP RMSE vs truth', 'Δ PIP RMSE vs truth'), ('F1 vs truth', '−Δ F1 vs truth'),
                ('95% interval score', 'Δ 95% interval score')]),
    'bodyfat': dict(
        label='bodyfat', replicates='optimisation seeds', vi_k=False,
        cols=[('PIP RMSE vs exact', 'PIP RMSE', 4), ('Mean excess marginal FKL vs exact', r'$\Delta\mathrm{FKL}$', 4),
              (K_COL, r'Mean $\widehat k$', 3), (ESS_COL, 'Mean ESS', 0),
              ('Exact posterior mass in 95% latent interval', 'Coverage', 4),
              ('Exact-posterior expected 95% interval score', 'Expected interval score', 4)],
        sensitivity=[('PIP RMSE vs exact', 'PIP RMSE'), ('Mean excess marginal FKL vs exact', 'Excess marginal KL'),
                     ('Exact posterior mass in 95% latent interval', 'Coverage'),
                     ('Exact-posterior expected 95% interval score', 'Interval score')],
        all_metrics='all_configuration_metrics.csv', evaluated='evaluated_configurations.csv',
        selected_training='selected_training_alpha_iterations_runtime.csv',
        paired=[('PIP RMSE vs exact', 'Δ PIP RMSE vs exact'), ('Mean excess marginal FKL vs exact', 'Δ Mean marginal KL vs exact'),
                ('Exact-posterior expected 95% interval score', 'Δ Exact-posterior expected 95% interval score')]),
    'riboflavin': dict(
        label='riboflavin', replicates='train--test splits', vi_k=False,
        cols=[('Predictive RMSE', 'RMSE', 4), ('MLPD', 'MLPD', 4), (K_COL, r'Mean $\widehat k$', 3),
              (ESS_COL, 'Mean ESS', 0), ('95% predictive coverage', 'Coverage', 4), ('95% interval score', 'Interval score', 4)],
        sensitivity=[('Predictive RMSE', 'Predictive RMSE'), ('MLPD', 'MLPD'),
                     ('95% predictive coverage', 'Coverage'), ('95% interval score', 'Interval score')],
        all_metrics='all_configuration_metrics.csv', evaluated='evaluated_configurations.csv',
        selected_training='selected_training_alpha_iterations_runtime.csv',
        paired=[('Predictive RMSE', 'Δ RMSE'), ('MLPD', '−Δ MLPD'), ('95% interval score', 'Δ 95% interval score')]),
}


def _fmt(value, decimals):
    if not np.isfinite(value):
        return '--'
    return f'{value:.{decimals}f}' if decimals else f'{value:.0f}'


def _finite_mean(x):
    x = pd.to_numeric(pd.Series(x), errors='coerce').to_numpy(float)
    ok = np.isfinite(x)
    return (float(x[ok].mean()) if ok.any() else np.nan,
            float(x[ok].std(ddof=1)) if ok.sum() > 1 else np.nan, int(ok.sum()))


def _write_tabular(lines_body, header, ncols, path, note):
    lines = [r'\thesistablestyle',
             r'\begin{tabular*}{\linewidth}{@{\extracolsep{\fill}}ll' + 'r' * (ncols - 2) + '@{}}',
             r'\toprule', ' & '.join(header) + r'\\', r'\midrule'] + lines_body
    lines += [r'\bottomrule', r'\end{tabular*}']
    if note:
        lines += [r'\par\medskip', r'\begin{minipage}{\linewidth}\footnotesize', 'Note. ' + note, r'\end{minipage}']
    Path(path).write_text('\n'.join(lines) + '\n', encoding='utf-8')


# ----------------------------------------------------------------------------- main table
def main_results_table(setting, run, out):
    spec = SETTINGS[setting]
    data = pd.read_csv(run / 'selected_configuration_metrics.csv')
    if ESS_COL not in data:
        raise KeyError(f'{ESS_COL} is missing from selected_configuration_metrics.csv')
    rows = []
    for stage in TABLE_STAGES:
        for method in TABLE_METHODS:
            g = data[(data.stage == stage) & (data.method == method)]
            if g.empty:
                raise ValueError(f'No selected results for {method} / {stage}')
            row = dict(stage=stage, method=method, replicates=g.seed.nunique())
            for col, _, _ in spec['cols']:
                source = g
                if col == K_COL and stage == 'Original VI' and spec['vi_k']:
                    source = data[(data.stage == 'Original PSIS') & (data.method == method)]
                mean, sd, n = _finite_mean(source[col])
                if stage == 'Original VI' and (col == ESS_COL or (col == K_COL and not spec['vi_k'])):
                    mean, sd, n = np.nan, np.nan, 0
                row[col], row[col + ' SD'], row[col + ' n_defined'] = mean, sd, n
            rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(out / f'{setting}_main_results_with_ESS.csv', index=False)
    total = int(data.seed.nunique())
    body = []
    for i, stage in enumerate(TABLE_STAGES):
        block = table[table.stage == stage]
        for j, method in enumerate(block.method):
            cells = []
            for col, _, dp in spec['cols']:
                value = block.loc[block.method == method, col].iloc[0]
                n = block.loc[block.method == method, col + ' n_defined'].iloc[0]
                cell = _fmt(value, dp)
                if col == K_COL and np.isfinite(value) and n < total:
                    cell += f' ({int(n)})'
                cells.append(cell)
            first = (r'\multirow{5}{*}{' + STAGE_TEX[stage] + '}') if j == 0 else ''
            body.append(f'{first} & {METHOD_TEX[method]} & ' + ' & '.join(cells) + r'\\')
        if i < len(TABLE_STAGES) - 1:
            body.append(r'\midrule')
    header = ['Procedure', 'Method'] + [h for _, h, _ in spec['cols']]
    note = (('Original VI and Ordinary PSIS share the VI proposal and therefore the same $\\widehat k$. '
             if spec['vi_k'] else 'The $\\widehat k$ under Ordinary PSIS describes the Original VI proposal. ')
            + f'ESS is averaged over the {total} {spec["replicates"]}; Original VI has no importance weights. '
            + f'Parentheses give the number of defined $\\widehat k$ values when fewer than {total}.')
    _write_tabular(body, header, len(header), out / f'{setting}_main_results_with_ESS.tex', note)
    return table


# ----------------------------------------------------------------------------- selected training records
def selected_training_table(setting, run, out):
    spec = SETTINGS[setting]
    d = pd.read_csv(run / spec['selected_training'])
    iters = next(c for c in ('iterations', 'iters', 'training_iterations') if c in d)
    time_col = next(c for c in ('training_runtime_seconds', 'training_time_seconds', 'time (s)') if c in d)
    rows = []
    for method in TABLE_METHODS:
        g = d[d.method == method]
        if g.empty:
            continue
        if method == 'ELBO':
            alpha = np.nan
        elif 'annealed' in method:
            alpha = pd.to_numeric(g.final_alpha, errors='coerce').mean()
        else:
            alpha = pd.to_numeric(g.alpha_setting, errors='coerce').mean()
        rows.append(dict(method=method, n=len(g), mean_alpha=alpha,
                         mean_iterations=pd.to_numeric(g[iters]).mean(),
                         mean_selected_runtime_seconds=pd.to_numeric(g[time_col]).mean()))
    table = pd.DataFrame(rows)
    table.to_csv(out / f'{setting}_selected_alpha_iterations_runtime.csv', index=False)
    return table


# ----------------------------------------------------------------------------- learning-rate table
def _main_allocation(df):
    for c in ['seed', 'learning_rate', 'S', 'N']:
        df[c] = pd.to_numeric(df[c], errors='raise')
    is_elbo = df['method'].eq('ELBO')
    return df.loc[(is_elbo & df['S'].eq(300) & df['N'].eq(1)) | (~is_elbo & df['S'].eq(1) & df['N'].eq(300))].copy()


def learning_rate_table(setting, run, out):
    df = _main_allocation(pd.read_csv(run / SETTINGS[setting]['evaluated']))
    value = 'evaluated_elbo_2000'
    df['_other'] = list(zip(df['alpha_setting'].astype(str), df['S'].astype(int), df['N'].astype(int)))
    if df.duplicated(['method', 'seed', 'learning_rate', '_other']).any():
        raise ValueError('Duplicate tuning configurations found.')
    for method, group in df.groupby('method'):
        grids = [frozenset(zip(g['seed'], g['_other'])) for _, g in group.groupby('learning_rate')]
        if any(grid != grids[0] for grid in grids[1:]):
            raise ValueError(f'{method}: unbalanced learning-rate grid')
    per_replicate = df.groupby(['method', 'seed', 'learning_rate'], as_index=False)[value].mean()
    summary = per_replicate.groupby(['method', 'learning_rate'], as_index=False)[value].mean()
    summary['best'] = summary[value].eq(summary.groupby('method')[value].transform('max'))
    summary.to_csv(out / f'{setting}_learning_rate_average_evaluated_ELBO.csv', index=False)
    best = summary[summary.best][['method', 'learning_rate', value]]
    best.to_csv(out / f'{setting}_best_learning_rate.csv', index=False)
    return best


# ----------------------------------------------------------------------------- configuration sensitivity
def configuration_sensitivity_table(setting, run, out):
    spec = SETTINGS[setting]
    d = pd.read_csv(run / spec['all_metrics'])
    d = d[d.stage.eq('Original VI')]
    if 'S' in d:
        d = _main_allocation(d)
    rows = []
    for method in TABLE_METHODS:
        g = d[d.method.eq(method)]
        row = dict(method=method)
        for col, name in spec['sensitivity']:
            x = g[['seed', col]].copy()
            x[col] = pd.to_numeric(x[col], errors='coerce')
            by_seed = x.groupby('seed')[col].agg(['mean', lambda v: v.std(ddof=1)])
            by_seed.columns = ['mean', 'sd']
            row[name] = by_seed['mean'].mean()
            row[name + ' (configuration SD)'] = by_seed['sd'].mean()
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(out / f'{setting}_original_VI_configuration_sensitivity.csv', index=False)
    return table


# ----------------------------------------------------------------------------- figures
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
LABEL = {'Damped PSIS': 'Iterative Damped PSIS'}
PAIRED_COLS = ['oriented_mean_difference_left_is_better', 'oriented_95% interval lower', 'oriented_95% interval upper']


def _save(fig, out, name):
    for ext in ['png', 'pdf']:
        fig.savefig(out / f'{name}.{ext}', dpi=300, bbox_inches='tight', pad_inches=.12)
    plt.close(fig)


def _clean_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)


def paired_plot(setting, run, out, only=False):
    specs = SETTINGS[setting]['paired']
    df = pd.read_csv(run / 'paired_comparisons_vs_ELBO.csv')
    df.stage = df.stage.replace({'Iterative Damped PSIS': 'Damped PSIS'})
    stages = ['Damped PSIS'] if only else TABLE_STAGES
    if df.duplicated(['method', 'stage', 'metric']).any():
        raise ValueError('Duplicate paired rows.')
    if len(specs) == 5:
        fig = plt.figure(figsize=(16.2, 8.8)); grid = fig.add_gridspec(2, 6, hspace=.55, wspace=.55)
        axes = [fig.add_subplot(grid[0, :3]), fig.add_subplot(grid[0, 3:]), fig.add_subplot(grid[1, :2]),
                fig.add_subplot(grid[1, 2:4]), fig.add_subplot(grid[1, 4:])]
    else:
        fig, axes = plt.subplots(1, 3, figsize=(16.2, 5.6))
    for i, (ax, (metric, title)) in enumerate(zip(axes, specs)):
        bounds = []
        for stage in stages:
            j = TABLE_STAGES.index(stage)
            dd = df[(df.metric == metric) & (df.stage == stage)].set_index('method').loc[TABLE_METHODS[1:]]
            x, lo, hi = dd[PAIRED_COLS].apply(pd.to_numeric, errors='raise').to_numpy(float).T
            good = np.isfinite(x) & np.isfinite(lo) & np.isfinite(hi)
            y = np.arange(4) + (0 if only else [-.27, -.09, .09, .27][j])
            color = '#2ca02c' if only else COLORS[j]
            ax.hlines(y[good], lo[good], hi[good], color=color, lw=1.1)
            ax.plot(x[good], y[good], linestyle='none', marker='D' if only else ['o', 's', '^', 'D'][j],
                    color=color, ms=5.3, label=LABEL.get(stage, stage))
            ax.vlines(lo[good], y[good] - .035, y[good] + .035, color=color, lw=1)
            ax.vlines(hi[good], y[good] - .035, y[good] + .035, color=color, lw=1)
            bounds.extend(np.r_[lo[good], hi[good], x[good]])
        bound = max(np.max(np.abs(bounds)) * 1.12 if bounds else .001, 1e-8)
        ax.set_xlim(-bound, bound); ax.axvline(0, color='.35', ls='--', lw=1)
        ax.set_yticks(range(4), TABLE_METHODS[1:]); ax.set_ylim(3.55, -.55)
        ax.set_title(f'({chr(97 + i)}) {title}', loc='left', fontsize=10)
        ax.set_xlabel('Paired difference vs ELBO'); _clean_axes(ax)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5)); ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
        if i not in ([0, 2] if len(specs) == 5 else [0]):
            ax.tick_params(axis='y', labelleft=False)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=len(stages), frameon=False, bbox_to_anchor=(.5, .025))
    fig.subplots_adjust(left=.11, right=.99, top=.88 if len(specs) == 5 else .80,
                        bottom=.16 if len(specs) == 5 else .20, wspace=.38)
    _save(fig, out, f'{setting}-paired' + ('-iterative-damped-only' if only else ''))


def _p1000_pips(run, gibbs_dir):
    selected = pd.read_csv(run / 'selected_configurations.csv'); rows = []
    for seed, g in selected.groupby('seed'):
        seed = int(seed); meta = pd.read_csv(gibbs_dir / f'runtime_and_metrics_gibbs_seed_{seed:02d}.csv').iloc[0]
        refs = []
        for chain in range(1, int(meta.n_chains) + 1):
            with np.load(gibbs_dir / f'gibbs_draws_seed_{seed:02d}_chain_{chain}.npz', allow_pickle=False) as z:
                refs.append(z['samples'].mean(0))
        ref = np.mean(refs, axis=0)
        for m in g.method:
            token = m.replace('*', 'star').replace('-', '_').replace(' ', '_').lower()
            d = pd.read_csv(run / 'selected_pips' / f'seed_{seed:02d}_{token}.csv')
            for s in TABLE_STAGES:
                rows.append(pd.DataFrame({'seed': seed, 'method': m, 'stage': s, 'reference_pip': ref,
                                          'method_pip': d['pip_' + s.lower().replace(' ', '_')]}))
    return pd.concat(rows, ignore_index=True)


def pip_scatter(setting, run, out, gibbs_dir=None, varbvs=None):
    if setting == 'p1000':
        pips = _p1000_pips(run, gibbs_dir)
    else:
        pips = pd.read_csv(run / 'selected_pips_all_stages.csv')
    pips.stage = pips.stage.replace({'Iterative Damped PSIS': 'Damped PSIS'})
    if varbvs is not None:
        pips = pd.concat([pips, varbvs], ignore_index=True)
    ref = 'exact' if setting == 'bodyfat' else 'Gibbs'
    for stage in TABLE_STAGES:
        df = pips[pips.stage.eq(stage)]
        methods = TABLE_METHODS + (['varbvs'] if stage == 'Original VI' and 'varbvs' in set(df.method) else [])
        fig, axes = plt.subplots(2, 3, figsize=(15.5, 9.2))
        for ax, m in zip(axes.flat, methods):
            dd = df[df.method.eq(m)]
            ax.scatter(dd.reference_pip, dd.method_pip, s=14, alpha=.38, color=COLORS[0])
            ax.plot([0, 1], [0, 1], '--', lw=1, color=COLORS[0])
            ax.set(xlim=(-.03, 1.03), ylim=(-.03, 1.03), title=m,
                   xlabel=f'{"Exact" if ref == "exact" else ref} posterior inclusion probability',
                   ylabel='Estimated posterior inclusion probability')
            _clean_axes(ax)
        for ax in list(axes.flat)[len(methods):]:
            ax.axis('off')
        fig.tight_layout(rect=[0, .02, 1, .97])
        suffix = '' if stage == 'Original VI' else '-' + LABEL.get(stage, stage).lower().replace(' ', '-')
        _save(fig, out, setting + '-pips' + suffix)
    # Figure 4.2: annealed VIMCO-GM across the four stages
    sub = pips[pips.method.eq('VIMCO-GM annealed')]
    fig, axes = plt.subplots(2, 2, figsize=(12.8, 10.0))
    for ax, stage in zip(axes.flat, ['Original VI', 'Original PSIS', 'Damped PSIS', 'IWMM PSIS']):
        dd = sub[sub.stage.eq(stage)]
        ax.scatter(dd.reference_pip, dd.method_pip, s=14, alpha=.38, color=COLORS[0])
        ax.plot([0, 1], [0, 1], '--', lw=1, color=COLORS[0])
        ax.set(xlim=(-.03, 1.03), ylim=(-.03, 1.03), title=stage,
               xlabel=f'{"Exact" if ref == "exact" else ref} posterior inclusion probability',
               ylabel='Estimated posterior inclusion probability')
        _clean_axes(ax)
    fig.tight_layout(rect=[0, .02, 1, .97])
    _save(fig, out, setting + '-postprocessing-pips')


# ----------------------------------------------------------------------------- p = 30 extras
def _holm(pvalues):
    values = np.asarray(pvalues, dtype=float); result = np.full(len(values), np.nan)
    valid = np.where(np.isfinite(values))[0]; ranked = valid[np.argsort(values[valid])]; largest = 0.0
    for rank, idx in enumerate(ranked):
        largest = max(largest, min(1.0, float(values[idx]) * (len(ranked) - rank))); result[idx] = largest
    return result


def _repeated_measures_f(x):
    n, k = x.shape
    contrasts = x - x[:, :1]
    centered = contrasts - contrasts.mean(axis=1, keepdims=True)
    treatment = centered.mean(axis=0)
    ss_s = float(n * np.sum(treatment ** 2)); error = centered - treatment[None, :]
    ss_error = float(np.sum(error ** 2)); df_s, df_e = k - 1, (n - 1) * (k - 1)
    F_value = (ss_s / df_s) / (ss_error / df_e)
    cov = np.cov(centered, rowvar=False, ddof=1)
    eps = float(np.clip(np.trace(cov) ** 2 / (df_s * np.sum(cov * cov)), 1.0 / df_s, 1.0))
    return dict(datasets=n, F=float(F_value), df1_GG=eps * df_s, df2_GG=eps * df_e, GG_epsilon=eps,
                p_uncorrected=float(f_distribution.sf(F_value, df_s, df_e)),
                p_GG=float(f_distribution.sf(F_value, eps * df_s, eps * df_e)))


def _select_two_mcse(group):
    """Evaluated-ELBO rule of Section 4.2.1: among configurations within two combined MCSEs of
    the largest evaluated ELBO, take the one with the smallest MCSE."""
    e, m = 'evaluated_elbo_2000', 'evaluated_elbo_2000 MCSE'
    top = group.loc[group[e].idxmax()]
    tied = group[(top[e] - group[e]) <= 2 * np.sqrt(top[m] ** 2 + group[m] ** 2)]
    return tied.sort_values(m).iloc[0]


def allocation_f_tests(run, out):
    """Table B.1: for every dataset, method and allocation (S, N), the learning rate (and alpha for
    the fixed variants) is selected by the evaluated-ELBO / MCSE rule, exactly as in the main protocol.
    The evaluated ELBO of the selected fits is compared across the five allocations with a one-way
    repeated-measures ANOVA (Greenhouse-Geisser corrected, Holm across the four methods). The same
    test is reported for the PIP RMSE of the selected fits (Original VI)."""
    d = pd.read_csv(run / 'evaluated_configurations.csv')
    levels = [(1, 300), (3, 100), (6, 50), (10, 30), (30, 10)]
    d = d[d.method.isin(TABLE_METHODS[1:]) & d.S.isin([s for s, _ in levels])]
    selected = pd.DataFrame([_select_two_mcse(g) for _, g in d.groupby(['method', 'seed', 'S', 'N'])])
    metrics = pd.read_csv(run / 'all_configuration_metrics_full_grid.csv')
    metrics = metrics[metrics.stage.eq('Original VI')].drop_duplicates('config_id')
    selected = selected.merge(metrics[['config_id', 'PIP RMSE vs Gibbs']], on='config_id', how='left')
    selected.to_csv(out / 'p30_allocation_selected_fits.csv', index=False)
    tables = {}
    for value, stem in [('evaluated_elbo_2000', 'evaluated_ELBO'), ('PIP RMSE vs Gibbs', 'PIP_RMSE')]:
        means, tests = [], []
        for method in TABLE_METHODS[1:]:
            wide = selected[selected.method.eq(method)].pivot(index='seed', columns='S', values=value)
            wide = wide[[s for s, _ in levels]]
            tests.append(dict(method=method, **_repeated_measures_f(wide.to_numpy(float))))
            means.append(dict(method=method, **{f'S={s}, N={n}': wide[s].mean() for s, n in levels}))
        tests = pd.DataFrame(tests); tests['p_GG_Holm'] = _holm(tests.p_GG)
        table = pd.DataFrame(means).merge(tests[['method', 'F', 'df1_GG', 'df2_GG', 'p_GG_Holm']], on='method')
        table.to_csv(out / f'p30_allocation_selected_{stem}_and_F_tests.csv', index=False)
        tables[stem] = table
    return tables


def varbvs_summary(path, out):
    """Table 4.3 varbvs row: PIP RMSE and excess marginal forward KL against the
    varbvs-matched Gibbs reference, averaged over the 50 datasets."""
    d = pd.read_csv(path)
    eps = 1e-12
    z = d.gibbs_pip.clip(eps, 1 - eps); q = d.varbvs_pip.clip(eps, 1 - eps)
    d['kl'] = z * np.log(z / q) + (1 - z) * np.log((1 - z) / (1 - q))
    per = d.groupby('replicate').agg(rmse=('squared_pip_error', lambda v: np.sqrt(v.mean())), fkl=('kl', 'mean'))
    per.to_csv(out / 'p30_varbvs_vs_matched_gibbs_by_replicate.csv')
    result = pd.DataFrame([dict(method='varbvs', **{'PIP RMSE': per.rmse.mean(), 'Delta FKL': per.fkl.mean()})])
    result.to_csv(out / 'p30_varbvs_row.csv', index=False)
    scatter = pd.DataFrame({'seed': d.replicate, 'method': 'varbvs', 'stage': 'Original VI',
                            'reference_pip': d.gibbs_pip, 'method_pip': d.varbvs_pip})
    return result, scatter


# ----------------------------------------------------------------------------- driver
def make_thesis_outputs(setting, results_dir, gibbs_dir=None, varbvs_csv=None):
    matplotlib.rcParams['text.usetex'] = False
    run = Path(results_dir).resolve()
    out = run / 'THESIS_TABLES_AND_FIGURES'; out.mkdir(exist_ok=True)
    produced = {}
    produced['main'] = main_results_table(setting, run, out)
    produced['training'] = selected_training_table(setting, run, out)
    produced['learning_rate'] = learning_rate_table(setting, run, out)
    produced['sensitivity'] = configuration_sensitivity_table(setting, run, out)
    varbvs_scatter = None
    if setting == 'p30':
        produced['allocation'] = allocation_f_tests(run, out)
        if varbvs_csv is not None and Path(varbvs_csv).is_file():
            produced['varbvs'], varbvs_scatter = varbvs_summary(varbvs_csv, out)
        else:
            warnings.warn('varbvs comparison CSV not found; varbvs row and panel skipped.')
    paired_plot(setting, run, out)
    paired_plot(setting, run, out, only=True)
    if setting in ('p30', 'p1000', 'bodyfat'):
        pip_scatter(setting, run, out, gibbs_dir=gibbs_dir)
    (out / 'README.txt').write_text(__doc__)
    print('Thesis tables and figures saved in', out)
    return produced


In [ ]:
RESULTS_DIR = RESULTS['out']
THESIS_OUTPUTS = make_thesis_outputs('bodyfat', RESULTS_DIR)